In [ ]:
# ============================================================
# CELL 1 — Install all Python dependencies for the full pipeline
# ============================================================
import subprocess, sys

print("Installing Neo4j driver, GraphRAG, and OpenAI client ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "neo4j", "neo4j-graphrag[openai]", "openai",
                "xgboost", "numba"], check=True)

# pyg-lib MUST be installed BEFORE torch_geometric is ever imported.
import torch
ver = torch.__version__.split("+")[0]
cu  = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
print(f"torch {torch.__version__} | CUDA {cu} | "
      f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to A100!'}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyg-lib", "-f", f"https://data.pyg.org/whl/torch-{ver}+{cu}.html"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True)

# Mount Google Drive (needed for persistence + reading HI-Small files)
from google.colab import drive
drive.mount("/content/drive")

print("\n✅ Cell 1 complete. Proceed to Cell 2.")

Installing Neo4j driver, GraphRAG, and OpenAI client ...
torch 2.11.0+cu130 | CUDA cu130 | GPU: Tesla T4
Mounted at /content/drive

✅ Cell 1 complete. Proceed to Cell 2.


In [ ]:
import os, subprocess
print("Neo4j dir exists:", os.path.isdir("/content/neo4j"))
print("Neo4j binary exists:", os.path.exists("/content/neo4j/bin/neo4j"))
print("Log file exists:", os.path.exists("/content/neo4j/logs/neo4j.log"))
print("Backup on Drive:", os.path.exists("/content/drive/MyDrive/aml/neo4j_backup.tar.gz"))
r = subprocess.run("java -version 2>&1", shell=True, capture_output=True, text=True)
print("Java:", (r.stdout + r.stderr).strip().splitlines()[0] if (r.stdout+r.stderr).strip() else "none")

Neo4j dir exists: True
Neo4j binary exists: True
Log file exists: True
Backup on Drive: True
Java: [0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [ ]:
# ============================================================
# CELL 2 — Neo4j lifecycle (bulletproof, verbose, idempotent)
# ============================================================
import os, re, subprocess, time, socket, shutil, tarfile

# ---------- Config ----------
DRIVE_ROOT     = "/content/drive/MyDrive"
AML_DIR        = f"{DRIVE_ROOT}/ibm aml"
RAW_CSV        = f"{AML_DIR}/HI-Small_Trans.csv"
STATE_DIR      = f"{DRIVE_ROOT}/aml/state"
NEO4J_BACKUP   = f"{DRIVE_ROOT}/aml/neo4j_backup.tar.gz"
NEO4J_TARBALL  = f"{DRIVE_ROOT}/aml/neo4j-5.20.0-unix.tar.gz"   # cached download
OUTPUT_DIR     = f"{DRIVE_ROOT}/aml/aml_outputs"

NEO4J_VERSION  = "5.20.0"
NEO4J_DIR      = "/content/neo4j"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "amlpassword123"

for d in (STATE_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

# ---------- Helpers ----------
def sh(cmd, timeout=180):
    """Run shell cmd, print its output, return CompletedProcess."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"   ⏱ timeout after {timeout}s: {cmd[:80]}")
        return subprocess.CompletedProcess(cmd, -1, "", "timeout")
    for line in (r.stdout or "").strip().splitlines()[:5]:
        print(f"   {line}")
    for line in (r.stderr or "").strip().splitlines()[:5]:
        print(f"   ! {line}")
    return r

def port_open(p=7687, t=2):
    try:
        socket.create_connection(("127.0.0.1", p), timeout=t).close()
        return True
    except OSError:
        return False

def java_version():
    r = subprocess.run("java -version 2>&1", shell=True, capture_output=True, text=True)
    m = re.search(r'version "(\d+)', r.stdout + r.stderr)
    return int(m.group(1)) if m else None

# ---------- 1. Java 17 or 21 ----------
print("1/7 Java")
ver = java_version()
print(f"   current: {ver}")
if ver not in (17, 21):
    print("   installing Java 17 …")
    sh("dpkg --configure -a", timeout=60)
    for lk in ("/var/lib/dpkg/lock-frontend", "/var/lib/dpkg/lock",
               "/var/cache/apt/archives/lock"):
        if os.path.exists(lk):
            subprocess.run(f"fuser -k {lk} 2>/dev/null || true",
                           shell=True, capture_output=True)
            try: os.remove(lk)
            except OSError: pass
    sh("DEBIAN_FRONTEND=noninteractive apt-get update -qq && "
       "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-17-jre-headless",
       timeout=240)
    jvm_dir = next((c for c in ("/usr/lib/jvm/java-17-openjdk-amd64",
                                "/usr/lib/jvm/java-17-openjdk")
                    if os.path.isdir(c)), None)
    if jvm_dir:
        sh(f"update-alternatives --install /usr/bin/java java {jvm_dir}/bin/java 2000", timeout=30)
        sh(f"update-alternatives --set java {jvm_dir}/bin/java", timeout=30)
    ver = java_version()
    print(f"   after install: {ver}")
if ver not in (17, 21):
    raise RuntimeError(f"Java {ver} unsupported. Neo4j 5.20 needs 17 or 21.\n"
                       f"  Manual: !ls /usr/lib/jvm/  then !update-alternatives --config java")
print("   ✅ Java ready\n")

# ---------- 2. Neo4j installed (Drive-cached tarball) ----------
print("2/7 Neo4j install")
if os.path.exists(f"{NEO4J_DIR}/bin/neo4j"):
    print("   ✅ already present")
else:
    if os.path.isdir(NEO4J_DIR):
        print(f"   removing incomplete {NEO4J_DIR}"); shutil.rmtree(NEO4J_DIR)
    if not os.path.exists(NEO4J_TARBALL):
        print(f"   downloading Neo4j {NEO4J_VERSION} → Drive cache …")
        r = sh(f"wget -q https://dist.neo4j.org/neo4j-community-{NEO4J_VERSION}-unix.tar.gz "
               f"-O {NEO4J_TARBALL}", timeout=180)
        if r.returncode != 0 or not os.path.exists(NEO4J_TARBALL):
            raise RuntimeError("download failed — check internet / Drive space")
    else:
        print(f"   using Drive cache: {NEO4J_TARBALL}")
    print("   extracting …")
    sh(f"tar -xzf {NEO4J_TARBALL} -C /content && "
       f"mv /content/neo4j-community-{NEO4J_VERSION} {NEO4J_DIR}", timeout=120)
    if not os.path.exists(f"{NEO4J_DIR}/bin/neo4j"):
        raise RuntimeError("extract failed — Neo4j binary missing")
    print("   ✅ installed")

# Config (append once)
conf = f"{NEO4J_DIR}/conf/neo4j.conf"
txt = open(conf).read()
if "# COLAB OVERRIDES" not in txt:
    mem_gb = int(open("/proc/meminfo").readline().split()[1]) / 1024 / 1024
    heap  = "3G" if mem_gb > 30 else "2G"
    cache = "1G" if mem_gb > 30 else "512M"
    with open(conf, "a") as f:
        f.write(f"\n# COLAB OVERRIDES\n"
                f"server.bolt.listen_address=:7687\n"
                f"server.http.listen_address=:7474\n"
                f"server.memory.heap.initial_size={heap}\n"
                f"server.memory.heap.max_size={heap}\n"
                f"server.memory.pagecache.size={cache}\n"
                f"dbms.security.allow_csv_import_from_file_urls=true\n")
    print(f"   ✅ config appended (heap={heap}, pagecache={cache})")

# Initial password
if not os.path.exists(f"{NEO4J_DIR}/data/dbms/auth.ini"):
    sh(f"cd {NEO4J_DIR} && bin/neo4j-admin dbms set-initial-password {NEO4J_PASSWORD}", timeout=60)
    print("   ✅ initial password set")
print()

# ---------- 3. Restore data from Drive if local data missing ----------
print("3/7 Data")
if os.path.isdir(f"{NEO4J_DIR}/data/databases/neo4j"):
    print("   ✅ data already present")
elif os.path.exists(NEO4J_BACKUP):
    print(f"   restoring {NEO4J_BACKUP} …")
    for d in ("data", "logs"):
        p = f"{NEO4J_DIR}/{d}"
        if os.path.isdir(p): shutil.rmtree(p)
    r = sh(f"cd {NEO4J_DIR} && tar -xzf {NEO4J_BACKUP}", timeout=180)
    if r.returncode == 0:
        print("   ✅ restored")
    else:
        print("   ⚠️  restore failed — starting fresh")
else:
    print("   no backup — starting fresh (run Cell 3 to load graph)")
print()

# ---------- 4. Cleanup stale state ----------
print("4/7 Cleanup")
r = subprocess.run("pgrep -f 'org.neo4j' 2>/dev/null",
                   shell=True, capture_output=True, text=True)
for pid in r.stdout.split():
    print(f"   killing orphan Neo4j JVM pid={pid}")
    subprocess.run(f"kill -9 {pid} 2>/dev/null", shell=True, capture_output=True)
for p in (f"{NEO4J_DIR}/run/neo4j.pid",
          f"{NEO4J_DIR}/data/neo4j.pid",
          f"{NEO4J_DIR}/data/dbms/store_lock",
          f"{NEO4J_DIR}/data/databases/store_lock"):
    if os.path.exists(p):
        try: os.remove(p); print(f"   removed {p}")
        except OSError as e: print(f"   could not remove {p}: {e}")
if not os.path.isdir(f"{NEO4J_DIR}/run"):
    os.makedirs(f"{NEO4J_DIR}/run", exist_ok=True)
print()

# ---------- 5. Start ----------
print("5/7 Start")
started = False
if port_open():
    print("   ✅ already listening")
    started = True
else:
    sh(f"{NEO4J_DIR}/bin/neo4j stop", timeout=60)      # no-op if not running
    r = subprocess.run(f"{NEO4J_DIR}/bin/neo4j start",
                       shell=True, capture_output=True, text=True, timeout=60)
    print(f"   neo4j start exit={r.returncode}")
    if (r.stdout or "").strip(): print(f"   {r.stdout.strip()}")
    if (r.stderr or "").strip(): print(f"   ! {r.stderr.strip()}")

    # Fast-fail: check for pid death while waiting for port
    pid_file = f"{NEO4J_DIR}/run/neo4j.pid"
    for i in range(60):
        if port_open():
            print(f"   ✅ port 7687 open after ~{i*2}s")
            started = True
            break
        if i >= 2 and os.path.exists(pid_file):
            pid = open(pid_file).read().strip()
            if pid and subprocess.run(f"ps -p {pid}",
                                       shell=True, capture_output=True).returncode != 0:
                print(f"   ❌ pid {pid} died while waiting — dumping logs")
                sh(f"tail -25 {NEO4J_DIR}/logs/neo4j.log")
                sh(f"tail -25 {NEO4J_DIR}/logs/debug.log")
                break
        time.sleep(2)

    if not started:
        # Fallback: run in foreground to capture the real error
        print("\n   ⚠️  daemon start didn't bring the port up. "
              "Running `neo4j console` for 15s to reveal the error:\n")
        import threading
        proc = subprocess.Popen([f"{NEO4J_DIR}/bin/neo4j", "console"],
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, start_new_session=True)
        def _reader():
            for line in proc.stdout:
                print(f"   > {line.rstrip()}")
        threading.Thread(target=_reader, daemon=True).start()
        time.sleep(15)
        if proc.poll() is None:
            proc.terminate()
            try: proc.wait(timeout=5)
            except subprocess.TimeoutExpired: proc.kill()
        # Console mode may have left pid/lock files behind
        for p in (f"{NEO4J_DIR}/run/neo4j.pid",
                  f"{NEO4J_DIR}/data/dbms/store_lock"):
            if os.path.exists(p):
                try: os.remove(p)
                except OSError: pass
        raise RuntimeError("Neo4j failed to start — see console output above for the real cause")
print()

# ---------- 6. Wait for the database itself to be online ----------
print("6/7 Waiting for database 'neo4j' to come online")
from neo4j import GraphDatabase
driver = GraphDatabase.driver(
    "bolt://127.0.0.1:7687",
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    keep_alive=True,
    max_connection_lifetime=300,
    connection_acquisition_timeout=10,
)

db_online = False
for i in range(30):
    try:
        with driver.session(database="system") as s:
            rec = s.run("SHOW DATABASES YIELD name, currentStatus "
                        "WHERE name = 'neo4j' RETURN currentStatus").single()
            if rec and rec["currentStatus"] == "online":
                print(f"   ✅ database online after ~{i*2}s")
                db_online = True
                break
    except Exception:
        pass
    time.sleep(2)
if not db_online:
    print("   ⚠️  database didn't report online in 60s — continuing (queries may fail)")
print()

# ---------- 7. Verify + helper ----------
print("7/7 Verify")
driver.verify_connectivity()
with driver.session() as s:
    n_acc = s.run("MATCH (a:Account) RETURN count(a) AS c").single()["c"]
    n_tx  = s.run("MATCH ()-[t:TRANSACTION]->() RETURN count(t) AS c").single()["c"]
print(f"   Accounts: {n_acc:,}  |  Transactions: {n_tx:,}")
if n_acc == 0:
    print("   ⚠️  graph is EMPTY — run Cell 3 to load HI-Small_Trans.csv")

def run_cypher(q, **params):
    with driver.session() as s:
        return list(s.run(q, **params))

print(f"\n✅ Cell 2 complete — Neo4j on bolt://127.0.0.1:7687 "
      f"({'loaded' if n_acc else 'empty graph, run Cell 3'})")

1/7 Java
   current: 21
   ✅ Java ready

2/7 Neo4j install
   ✅ already present

3/7 Data
   ✅ data already present

4/7 Cleanup
   killing orphan Neo4j JVM pid=5177
   killing orphan Neo4j JVM pid=5840
   removed /content/neo4j/run/neo4j.pid
   removed /content/neo4j/data/databases/store_lock

5/7 Start
   ✅ already listening

6/7 Waiting for database 'neo4j' to come online


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 3 — Load HI-Small_Trans.csv into Neo4j (idempotent)
# ============================================================
import pandas as pd, urllib.parse, tarfile, time

def _graph_size():
    r = run_cypher("MATCH (a:Account)-[t:TRANSACTION]->(b:Account) "
                   "RETURN count(t) AS tx, sum(CASE WHEN t.label=1 THEN 1 ELSE 0 END) AS lx")[0]
    return r["tx"], int(r["lx"] or 0)

if os.path.exists(FLAG_NEO4J):
    tx, lx = _graph_size()
    print(f"✅ Graph already loaded ({tx:,} tx | {lx:,} laundering). Skipping.")
else:
    # 1. Preprocess — flattens the duplicate "Account" column problem
    RAW_NAMES = ["Timestamp","From Bank","Account","To Bank","Account.1",
                 "Amount Received","Receiving Currency","Amount Paid",
                 "Payment Currency","Payment Format","Is Laundering"]
    print("Reading raw IBM CSV ...")
    df = pd.read_csv(RAW_CSV, header=0, names=RAW_NAMES,
                     dtype={"From Bank":str,"Account":str,"To Bank":str,"Account.1":str})
    print(f"  {len(df):,} rows | laundering {int(df['Is Laundering'].sum()):,}")
    df["source"] = df["From Bank"] + "_" + df["Account"]
    df["target"] = df["To Bank"]   + "_" + df["Account.1"]
    clean = df[["Timestamp","source","target","Amount Paid","Amount Received",
                "Payment Currency","Receiving Currency","Payment Format","Is Laundering"]].rename(columns={
        "Timestamp":"timestamp","Amount Paid":"amount_paid","Amount Received":"amount_received",
        "Payment Currency":"currency_paid","Receiving Currency":"currency_received",
        "Payment Format":"payment_format","Is Laundering":"label"})
    os.makedirs(f"{NEO4J_DIR}/import", exist_ok=True)
    clean.to_csv(f"{NEO4J_DIR}/import/hi_small_clean.csv", index=False)
    print(f"  wrote clean CSV ({os.path.getsize(f'{NEO4J_DIR}/import/hi_small_clean.csv')/1e6:.0f} MB)")

    # 2. Schema
    with driver.session() as s:
        s.run("CREATE CONSTRAINT account_id IF NOT EXISTS FOR (a:Account) REQUIRE a.id IS UNIQUE")
        s.run("CREATE INDEX tx_ts IF NOT EXISTS FOR ()-[t:TRANSACTION]-() ON (t.timestamp)")
        s.run("CREATE INDEX tx_lbl IF NOT EXISTS FOR ()-[t:TRANSACTION]-() ON (t.label)")

    # 3. Wipe any stale data
    with driver.session() as s:
        s.run("MATCH (n) DETACH DELETE n").consume()

    # 4. Load
    print("Loading into Neo4j (~5-8 min) ...")
    t0 = time.time()
    with driver.session() as s:
        s.run("""
        LOAD CSV WITH HEADERS FROM 'file:///hi_small_clean.csv' AS row
        CALL {
            WITH row
            MERGE (a:Account {id: row.source})
            MERGE (b:Account {id: row.target})
            CREATE (a)-[:TRANSACTION {
                amount_paid:       toFloat(row.amount_paid),
                amount_received:   toFloat(row.amount_received),
                currency_paid:     row.currency_paid,
                currency_received: row.currency_received,
                payment_format:    row.payment_format,
                timestamp:         row.timestamp,
                label:             toInteger(row.label)
            }]->(b)
        } IN TRANSACTIONS OF 10000 ROWS
        """).consume()
    print(f"  loaded in {time.time()-t0:.0f}s")

    # 5. Verify
    tx, lx = _graph_size()
    assert tx > 4_000_000, f"Load incomplete: only {tx:,} transactions"
    assert lx == int(df["Is Laundering"].sum()), f"Laundering mismatch: {lx:,} vs {int(df['Is Laundering'].sum()):,}"
    print(f"✅ Graph loaded: {tx:,} transactions | {lx:,} laundering")

    # 6. Backup to Drive so a reset restores instantly
    print("Backing up Neo4j data to Drive ...")
    subprocess.run(f"{NEO4J_DIR}/bin/neo4j stop", shell=True, capture_output=True)
    time.sleep(3)
    with tarfile.open(NEO4J_BACKUP, "w:gz") as tar:
        tar.add(f"{NEO4J_DIR}/data", arcname="data")
    subprocess.run(f"{NEO4J_DIR}/bin/neo4j start", shell=True, capture_output=True)
    for _ in range(60):
        if _bolt_ok(): break
        time.sleep(2)
    open(FLAG_NEO4J, "w").write("1")
    print(f"✅ Backup at {NEO4J_BACKUP} ({os.path.getsize(NEO4J_BACKUP)/1e6:.0f} MB)")

print("\n✅ Cell 3 complete.")

NameError: name 'FLAG_NEO4J' is not defined

In [ ]:
#!/usr/bin/env python
"""
AML FINAL PIPELINE  -  IBM HI-Small  |  causal features + GNN + XGBoost + validated blend
=====================================================================================

Cell 4 of the master AML pipeline. Runs the full detection stage on HI-Small_Trans.csv
and caches everything to Google Drive so a Colab reset only costs the in-progress step.
"""
import argparse
import copy
import glob
import hashlib
import json
import os
import shutil
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, precision_recall_curve
from sklearn.model_selection import KFold, cross_val_predict

try:
    from numba import njit
    HAVE_NUMBA = True
except Exception:
    HAVE_NUMBA = False

# ========================================================================================
# RUN SETTINGS
# ========================================================================================
# ★ CHANGE 1: read the RAW IBM file (not the old Neo4j-derived CSV) so account IDs match
#             Neo4j (Bank_Account) and the patterns file aligns natively.
CSV_PATH = "HI-Small_Trans.csv"
PATTERNS_PATH = "HI-Small_Patterns.txt"

# ★ CHANGE 2a: point DRIVE_DIR at the folder where the raw files actually live.
DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_DIR = DRIVE_ROOT + "/ibm aml"
LOCAL_DIR = "/content" if os.path.isdir("/content") else "."

# ★ CHANGE 2b: pin OUTPUT_DIR and CACHE_DIR to the same folders Cell 2 / Cell 5 use.
#              This is what makes results.json + explanations.json land in the same place
#              across resets and across cells.
OUTPUT_DIR = DRIVE_ROOT + "/aml/aml_outputs"
CACHE_DIR = DRIVE_ROOT + "/aml/feature_cache"

# ★ CHANGE 2c: state dir for the resume flags (Cell 2 writes neo4j_loaded.flag here;
#              this cell writes pipeline_done.flag; Cell 5 writes graphrag_done.flag).
STATE_DIR = DRIVE_ROOT + "/aml/state"
try:
    os.makedirs(STATE_DIR, exist_ok=True)
except OSError:
    pass  # not in Colab / Drive not mounted — harmless fallback

COPY_TO_LOCAL = True
CACHE_FEATURES = True
CACHE_GNN = True
FORCE_RECOMPUTE = False
USE_NEO4J_TEMPORAL = True
FEATURE_VERSION = "4"
_CFG = dict(csv=CSV_PATH, patterns=PATTERNS_PATH, out=OUTPUT_DIR)
USE_GNN = True
STACK_OOF = False
SMOKE = False
MOUNT_DRIVE = os.path.isdir("/content")
FEATURE_SETS = ["structural", "full"]
GNN_FEATURE_SETS = ["structural"]
AUDIT_FINGERPRINT_ONLY = True
SEED = 42

COL_SRC, COL_DST, COL_AMT, COL_TS, COL_LABEL = "source", "target", "amount", "timestamp", "label"
CAT_COLS = ["payment_format", "currency_paid"]
PATTERN_ID_MODE = "account"
INPUT_KIND = "unknown"

TRAIN_FRAC, VAL_FRAC = 0.60, 0.20
WINDOWS = [3600, 86400, 345600, 604800]
DISTINCT_OWN = [("24h", 86400), ("96h", 345600), ("all", None)]
DISTINCT_CROSS_WINDOW = 345600
MISSING_GAP = 10_000_000
BURN_THRESH, BURN_CAP = 0.02, 0.35

NUM_NEIGHBORS = [15, 10, 5, 5]
HIDDEN, LAYERS, DROPOUT = 64, 4, 0.2
EPOCHS, LR = 60, 2e-3
EVAL_EVERY = 10
NEG_RATIO, POS_WEIGHT = 20, 3.0
BATCH_TRAIN, BATCH_EVAL = 2048, 4096
USE_EGO_MARKERS = True
GNN_BURN_IN = False
GNN_FOLD_BOUNDS = [0.30, 0.45]
NUM_WORKERS = 0

SCALE_POS_WEIGHT = 30.0
THRESH_BETA = 1.0
N_BOOT = 200
ALERT_BUDGETS = (0.001, 0.005, 0.01)


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ========================================================================================
# 1. LOAD
# ========================================================================================
RAW_NAMES = ["Timestamp", "From Bank", "Account", "To Bank", "Account.1", "Amount Received",
             "Receiving Currency", "Amount Paid", "Payment Currency", "Payment Format", "Is Laundering"]


def to_seconds(s, allow_nan=False):
    s = s if isinstance(s, pd.Series) else pd.Series(s)
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype("float64").to_numpy()
        mx = np.nanmax(np.abs(v)) if np.isfinite(v).any() else 0.0
        v = v / (1e9 if mx > 1e17 else 1e6 if mx > 1e14 else 1e3 if mx > 1e11 else 1.0)
        out = np.floor(v)
    else:
        t = pd.to_datetime(s, utc=True)
        out = ((t - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1)).to_numpy(dtype="float64")
    return out if allow_nan else out.astype("int64")


def is_raw_ibm(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        head = fh.readline()
    return "Is Laundering" in head and "Payment Format" in head


def read_table(path):
    global PATTERN_ID_MODE, INPUT_KIND
    if is_raw_ibm(path):
        raw = pd.read_csv(path, header=0, names=RAW_NAMES,
                          dtype={"From Bank": str, "Account": str, "To Bank": str, "Account.1": str})
        df = pd.DataFrame({
            COL_SRC: raw["From Bank"] + "_" + raw["Account"],
            COL_DST: raw["To Bank"] + "_" + raw["Account.1"],
            COL_AMT: raw["Amount Paid"], "amount_received": raw["Amount Received"],
            COL_TS: raw["Timestamp"], COL_LABEL: raw["Is Laundering"],
            "payment_format": raw["Payment Format"], "currency_paid": raw["Payment Currency"],
            "currency_received": raw["Receiving Currency"],
            "source_bank": raw["From Bank"], "target_bank": raw["To Bank"]})
        PATTERN_ID_MODE = "bank_account"
        INPUT_KIND = "raw_ibm"
        log("  raw IBM file detected: payment format, currencies, banks and amounts are used")
        return df
    need = [COL_SRC, COL_DST, COL_AMT, COL_TS, COL_LABEL]
    cols = list(pd.read_csv(path, nrows=0).columns)
    missing = [c for c in need if c not in cols]
    if missing:
        raise ValueError(f"{os.path.basename(path)} has columns {cols}; missing {missing}.")
    use = need + [c for c in ("source_created", "target_created") if c in cols]
    df = pd.read_csv(path, usecols=use, dtype={COL_SRC: str, COL_DST: str})
    INPUT_KIND = "neo4j"
    log(f"  Neo4j export detected (columns used: {', '.join(use)}).")
    return df


def load_data(path):
    df = read_table(path)
    ts_abs = to_seconds(df[COL_TS])
    order = np.argsort(ts_abs, kind="stable")
    df = df.iloc[order].reset_index(drop=True)
    ts_abs = ts_abs[order]
    n = len(df)
    src_raw = df[COL_SRC].astype(str).to_numpy()
    dst_raw = df[COL_DST].astype(str).to_numpy()
    codes, uniques = pd.factorize(np.concatenate([src_raw, dst_raw]))
    src, dst = codes[:n].astype(np.int64), codes[n:].astype(np.int64)
    y = df[COL_LABEL].fillna(0).astype(int).to_numpy().astype(np.int64)
    amt = df[COL_AMT].fillna(0.0).clip(lower=0).to_numpy(dtype="float64")
    pos = np.where(y == 1)[0]
    created = None
    if USE_NEO4J_TEMPORAL and "source_created" in df.columns and "target_created" in df.columns:
        sc = to_seconds(df["source_created"], allow_nan=True)
        tc = to_seconds(df["target_created"], allow_nan=True)
        created = (np.where(np.isfinite(sc), sc, ts_abs).astype(np.int64),
                   np.where(np.isfinite(tc), tc, ts_abs).astype(np.int64))
    return dict(df=df, n=n, N=len(uniques), src=src, dst=dst, y=y, amt=amt, ts=ts_abs - ts_abs.min(),
                ts_abs=ts_abs, src_raw_pos=src_raw[pos], dst_raw_pos=dst_raw[pos], pos_idx=pos,
                created=created, created_ok=False)


def audit_created(d):
    if d.get("created") is None:
        return False
    sc, tc = d["created"]
    ts = d["ts_abs"]
    first = np.full(d["N"], np.iinfo(np.int64).max, dtype=np.int64)
    for codes in (d["src"], d["dst"]):
        u, idx = np.unique(codes, return_index=True)
        first[u] = np.minimum(first[u], ts[idx])
    fut_s, fut_t = float((sc > ts).mean()), float((tc > ts).mean())
    eq_s, eq_t = float((sc == first[d["src"]]).mean()), float((tc == first[d["dst"]]).mean())
    ok = fut_s == 0.0 and fut_t == 0.0
    log(f"  Neo4j temporal columns audit: source_created later in {fut_s:.2%}, target_created in {fut_t:.2%}")
    log(f"    equal to the account's first sighting: {eq_s:.1%} / {eq_t:.1%}")
    if not ok:
        log("    -> NOT causal: IGNORED")
        return False
    if eq_s >= 0.999 and eq_t >= 0.999:
        log("    -> causal but redundant: not added")
        return False
    log("    -> used as extra account-age features")
    return True


# ========================================================================================
# 2. CAUSAL FEATURES
# ========================================================================================
def build_stream(acct, ts, amt, big):
    key = acct.astype(np.int64) * big + ts
    order = np.argsort(key, kind="stable")
    cs = np.concatenate([[0.0], np.cumsum(amt[order], dtype=np.float64)])
    return key[order], cs, amt[order]


def hist_query(stream, q_acct, q_ts, big, window=None):
    key_s, cs, _ = stream
    base = q_acct.astype(np.int64) * big
    hi = np.searchsorted(key_s, base + q_ts, side="left")
    lo_key = base if window is None else base + np.maximum(q_ts - window, 0)
    lo = np.searchsorted(key_s, lo_key, side="left")
    return (hi - lo).astype(np.float32), (cs[hi] - cs[lo]).astype(np.float32)


def last_event(stream, q_acct, q_ts, big):
    key_s, _, amt_s = stream
    base = q_acct.astype(np.int64) * big
    hi = np.searchsorted(key_s, base + q_ts, side="left")
    lo = np.searchsorted(key_s, base, side="left")
    has = hi > lo
    idx = np.maximum(hi - 1, 0)
    gap = np.where(has, q_ts - (key_s[idx] - base), MISSING_GAP).astype(np.float32)
    return gap, np.where(has, amt_s[idx], 0.0), has


def first_seen(codes, ts, N):
    u, idx = np.unique(codes, return_index=True)
    out = np.full(N, MISSING_GAP, dtype=np.int64)
    out[u] = ts[idx]
    return out


def _distinct_sweep(seg_start, seg_end, ev_ts, ev_partner, q_acct, q_ts, nq, window, cnt, out):
    cur = -1
    lo = 0
    hi = 0
    distinct = 0
    for k in range(nq):
        a = q_acct[k]
        if a != cur:
            for j in range(lo, hi):
                cnt[ev_partner[j]] -= 1
            distinct = 0
            lo = seg_start[a]
            hi = seg_start[a]
            cur = a
        t = q_ts[k]
        e = seg_end[a]
        while hi < e and ev_ts[hi] < t:
            c = ev_partner[hi]
            if cnt[c] == 0:
                distinct += 1
            cnt[c] += 1
            hi += 1
        lim = t - window
        while lo < hi and ev_ts[lo] < lim:
            c = ev_partner[lo]
            cnt[c] -= 1
            if cnt[c] == 0:
                distinct -= 1
            lo += 1
        out[k] = distinct


_SWEEP_NB = njit(cache=False)(_distinct_sweep) if HAVE_NUMBA else None
_NUMBA_OK = {"ok": HAVE_NUMBA}


def stream_struct(acct, partner, ts, N):
    order = np.argsort(acct, kind="stable")
    a_s = acct[order]
    ids = np.arange(N)
    return (np.searchsorted(a_s, ids, "left").astype(np.int64), np.searchsorted(a_s, ids, "right").astype(np.int64),
            ts[order].astype(np.int64), partner[order].astype(np.int64))


def query_struct(q_acct, q_ts):
    order = np.argsort(q_acct, kind="stable")
    return order, q_acct[order].astype(np.int64), q_ts[order].astype(np.int64)


def distinct_counts(stream, qs, N, window):
    seg_start, seg_end, ev_ts, ev_partner = stream
    order, qa, qt = qs
    win = (1 << 60) if window is None else int(window)
    nq = len(qa)
    done = False
    if _NUMBA_OK["ok"]:
        try:
            out_s = np.zeros(nq, dtype=np.int64)
            _SWEEP_NB(seg_start, seg_end, ev_ts, ev_partner, qa, qt, nq, win, np.zeros(N, dtype=np.int64), out_s)
            done = True
        except Exception as e:
            log(f"  numba sweep failed ({type(e).__name__}); using the slower pure-Python path")
            _NUMBA_OK["ok"] = False
    if not done:
        out_l = [0] * nq
        _distinct_sweep(seg_start.tolist(), seg_end.tolist(), ev_ts.tolist(), ev_partner.tolist(),
                        qa.tolist(), qt.tolist(), nq, win, [0] * N, out_l)
        out_s = np.asarray(out_l, dtype=np.int64)
    out = np.empty(nq, dtype=np.float32)
    out[order] = out_s
    return out


def causal_features(d):
    src, dst, ts, amt, N, n = d["src"], d["dst"], d["ts"], d["amt"], d["N"], d["n"]
    big = int(ts.max()) + max(WINDOWS) + 2
    out_stream = build_stream(src, ts, amt, big)
    in_stream = build_stream(dst, ts, amt, big)
    cf, cr = src * N + dst, dst * N + src
    uniq = np.unique(np.concatenate([cf, cr]))
    pid_f, pid_r = np.searchsorted(uniq, cf), np.searchsorted(uniq, cr)
    pair_stream = build_stream(pid_f, ts, amt, big)

    f, logged = {}, set()

    def add(name, arr, log1p=False):
        f[name] = np.asarray(arr, dtype=np.float32)
        if log1p:
            logged.add(name)

    for pre, acct in (("s", src), ("r", dst)):
        for tag, stream in (("out", out_stream), ("in", in_stream)):
            c, s = hist_query(stream, acct, ts, big)
            add(f"{pre}_{tag}_cnt", c, True)
            add(f"{pre}_{tag}_sum", s, True)
            for w in WINDOWS:
                c, s = hist_query(stream, acct, ts, big, w)
                add(f"{pre}_{tag}_cnt_{w // 3600}h", c, True)
                add(f"{pre}_{tag}_sum_{w // 3600}h", s, True)
            gap, last_amt, has = last_event(stream, acct, ts, big)
            add(f"{pre}_gap_{tag}", gap, True)
            if pre == "s" and tag == "in":
                add("s_fwd_ratio", np.where(has, np.clip(amt / (last_amt + 1.0), 0, 10), 0.0))

    eps = 1.0
    add("s_imb", np.abs(f["s_in_sum"] - f["s_out_sum"]) / (f["s_in_sum"] + f["s_out_sum"] + eps))
    add("r_imb", np.abs(f["r_in_sum"] - f["r_out_sum"]) / (f["r_in_sum"] + f["r_out_sum"] + eps))
    for w in WINDOWS[1:]:
        h = f"{w // 3600}h"
        for p in ("s", "r"):
            i_, o_ = f[f"{p}_in_sum_{h}"], f[f"{p}_out_sum_{h}"]
            add(f"{p}_imb_{h}", np.abs(i_ - o_) / (i_ + o_ + eps))
            add(f"{p}_pass_{h}", np.clip(o_ / (i_ + eps), 0, 10))
    add("amt_vs_s_avg", np.clip(amt / (f["s_out_sum"] / np.maximum(f["s_out_cnt"], 1) + eps), 0, 50), True)
    add("amt_vs_r_avg", np.clip(amt / (f["r_in_sum"] / np.maximum(f["r_in_cnt"], 1) + eps), 0, 50), True)

    c, _ = hist_query(pair_stream, pid_f, ts, big)
    add("pair_fwd_cnt", c, True)
    c, _ = hist_query(pair_stream, pid_f, ts, big, 345600)
    add("pair_fwd_cnt_96h", c, True)
    c, s = hist_query(pair_stream, pid_r, ts, big)
    add("pair_rev_cnt", c, True)
    add("pair_rev_sum", s, True)
    c, _ = hist_query(pair_stream, pid_r, ts, big, 345600)
    add("pair_rev_cnt_96h", c, True)

    ev_out, ev_in = stream_struct(src, dst, ts, N), stream_struct(dst, src, ts, N)
    q_src, q_dst = query_struct(src, ts), query_struct(dst, ts)
    for tag, w in DISTINCT_OWN:
        add(f"s_out_nuniq_{tag}", distinct_counts(ev_out, q_src, N, w), True)
        add(f"r_in_nuniq_{tag}", distinct_counts(ev_in, q_dst, N, w), True)
    add("r_out_nuniq_96h", distinct_counts(ev_out, q_dst, N, DISTINCT_CROSS_WINDOW), True)
    add("s_in_nuniq_96h", distinct_counts(ev_in, q_src, N, DISTINCT_CROSS_WINDOW), True)

    first = np.minimum(first_seen(src, ts, N), first_seen(dst, ts, N))
    add("s_age", ts - first[src], True)
    add("r_age", ts - first[dst], True)
    if d.get("created_ok"):
        sc, tc = d["created"]
        add("s_created_age", np.maximum(d["ts_abs"] - sc, 0), True)
        add("r_created_age", np.maximum(d["ts_abs"] - tc, 0), True)
    extras = {"s_new": ((f["s_out_cnt"] + f["s_in_cnt"]) == 0),
              "r_new": ((f["r_out_cnt"] + f["r_in_cnt"]) == 0)}

    hour = (d["ts_abs"] % 86400) / 86400.0 * 2 * np.pi
    add("log_amount", amt, True)
    add("sin_hour", np.sin(hour))
    add("cos_hour", np.cos(hour))
    add("self_loop", src == dst)

    df = d["df"]
    if "amount_received" in df.columns:
        rec = df["amount_received"].fillna(0.0).clip(lower=0).to_numpy(dtype="float64")
        add("amt_equal", np.isclose(rec, amt, rtol=1e-6, atol=1e-6))
        add("amt_logratio", np.abs(np.log((rec + 1.0) / (amt + 1.0))))
    for col in CAT_COLS:
        if col in df.columns:
            codes, names_ = pd.factorize(df[col].astype(str))
            counts = np.bincount(codes[:int(TRAIN_FRAC * n)], minlength=len(names_))
            keep = sorted(range(len(names_)), key=lambda k: (-counts[k], str(names_[k])))[:15]
            for k in keep:
                add(f"{col}={names_[k]}", codes == k)
    if "source_bank" in df.columns and "target_bank" in df.columns:
        add("same_bank", (df["source_bank"].astype(str) == df["target_bank"].astype(str)).to_numpy())
    if "currency_paid" in df.columns and "currency_received" in df.columns:
        add("same_currency", (df["currency_paid"].astype(str) == df["currency_received"].astype(str)).to_numpy())

    names = list(f.keys())
    X = np.empty((n, len(names)), dtype=np.float32)
    for j, k in enumerate(names):
        v = f[k]
        X[:, j] = np.log1p(np.maximum(v, 0)) if k in logged else v
    return X, names, extras


def choose_burn_in(cold, tr_end, n):
    chunk = max(n // 100, 1)
    rates = [float(cold[i:i + chunk].mean()) for i in range(0, tr_end, chunk)]
    bad = [i for i, r in enumerate(rates) if r >= BURN_THRESH]
    burn_end = min(((bad[-1] + 1) * chunk) if bad else 0, int(BURN_CAP * tr_end))
    return burn_end, rates


# ========================================================================================
# 3. METRICS
# ========================================================================================
def best_threshold(y, p, beta):
    pre, rec, thr = precision_recall_curve(y, p)
    b2 = beta ** 2
    fb = (1 + b2) * pre[:-1] * rec[:-1] / (b2 * pre[:-1] + rec[:-1] + 1e-12)
    return float(thr[int(np.argmax(fb))])


def prf(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum())
    p = tp / max(int(pred.sum()), 1)
    r = tp / max(int(y.sum()), 1)
    return p, r, 2 * p * r / (p + r + 1e-12)


class Ranked:
    def __init__(self, y, p):
        self.order = np.argsort(-p, kind="stable")
        self.y = y[self.order].astype(np.float64)

    def ap(self, w):
        wo = w[self.order]
        pos = wo * self.y
        s = pos.sum()
        if s <= 0:
            return float("nan")
        return float((pos * np.cumsum(pos) / np.maximum(np.cumsum(wo), 1e-12)).sum() / s)


def boot_weights(n, rng, block=0):
    if block:
        nb = max(n // block, 1)
        starts = rng.integers(0, max(n - block, 1), nb)
        idx = (starts[:, None] + np.arange(block)[None, :]).ravel()
    else:
        idx = rng.integers(0, n, n)
    return np.bincount(idx, minlength=n).astype(np.float64)


def bootstrap_ap(y, p, n_boot, seed=0, block=0):
    rng, rk = np.random.default_rng(seed), Ranked(y, p)
    aps = [rk.ap(boot_weights(len(y), rng, block)) for _ in range(n_boot)]
    aps = [a for a in aps if a == a]
    return tuple(np.percentile(aps, [2.5, 97.5])) if aps else (float("nan"), float("nan"))


def paired_bootstrap_ap(y, p_a, p_b, n_boot, seed=0, block=0):
    rng, ra, rb = np.random.default_rng(seed), Ranked(y, p_a), Ranked(y, p_b)
    diffs = []
    for _ in range(n_boot):
        w = boot_weights(len(y), rng, block)
        a, b = ra.ap(w), rb.ap(w)
        if a == a and b == b:
            diffs.append(a - b)
    diffs = np.asarray(diffs)
    return float(diffs.mean()), tuple(np.percentile(diffs, [2.5, 97.5])), float((diffs > 0).mean())


def make_run(name, short, p_va, p_te, yva, yte, model=None, extra=None):
    thr = best_threshold(yva, p_va, THRESH_BETA)
    pr, rc, f1 = prf(yte, (p_te >= thr).astype(int))
    r = dict(name=name, short=short, ap_val=float(average_precision_score(yva, p_va)),
             ap_test=float(average_precision_score(yte, p_te)), thr=thr, precision=pr, recall=rc, f1=f1,
             p_va=p_va, p_te=p_te, model=model)
    if extra:
        r.update(extra)
    return r


def fit_xgb(name, short, Xtr, ytr, Xva, yva, Xte, yte):
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        device = "cpu"
    log(f"XGBoost [{name}]  train={len(ytr):,} (pos {int(ytr.sum()):,})  device={device}")
    clf = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.8,
        colsample_bytree=0.8, scale_pos_weight=SCALE_POS_WEIGHT, tree_method="hist", device=device,
        eval_metric="aucpr", early_stopping_rounds=100, random_state=SEED)
    clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=200)
    return make_run(name, short, clf.predict_proba(Xva)[:, 1], clf.predict_proba(Xte)[:, 1], yva, yte,
                    model=clf, extra=dict(best_iter=int(clf.best_iteration)))


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def blend_run(run_tab, g_va, g_te, yva, yte):
    Zv = np.column_stack([logit(run_tab["p_va"]), g_va])
    Zt = np.column_stack([logit(run_tab["p_te"]), g_te])
    mu, sd = Zv.mean(0), Zv.std(0) + 1e-9
    Zv, Zt = (Zv - mu) / sd, (Zt - mu) / sd
    lr = LogisticRegression(C=1.0, max_iter=1000)
    oof = cross_val_predict(lr, Zv, yva, cv=KFold(5, shuffle=True, random_state=SEED), method="decision_function")
    lr.fit(Zv, yva)
    w = lr.coef_[0]
    log(f"  blend weights: tabular {w[0]:+.2f} | GNN {w[1]:+.2f}")
    return make_run("blend (tabular + GNN, fitted on val)", "blend", oof, lr.decision_function(Zt), yva, yte,
                    extra=dict(weights=[float(w[0]), float(w[1])]))


# ========================================================================================
# 4. TYPOLOGIES
# ========================================================================================
def load_typologies(path, d):
    rows, cur = [], None
    with open(path, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):
                cur = line.split(" - ", 1)[1].split(":")[0].strip()
            elif line.startswith("END LAUNDERING ATTEMPT"):
                cur = None
            elif cur and line:
                p = line.split(",")
                rows.append((cur, p[0], p[1], p[2], p[3], p[4], float(p[5]), float(p[7])))
    pat = pd.DataFrame(rows, columns=["typology", "ts", "fb", "fa", "tb", "ta", "amt_recv", "amt_paid"])
    pat["ts"] = to_seconds(pd.to_datetime(pat["ts"], format="%Y/%m/%d %H:%M"))
    fmts = {"account": lambda b, a: a, "bank_account": lambda b, a: f"{b}_{a}",
            "bank+account": lambda b, a: f"{b}{a}", "account_bank": lambda b, a: f"{a}_{b}"}
    ds_ids = set(d["src_raw_pos"]) | set(d["dst_raw_pos"])
    best_mode, best_ov = "account", -1.0
    for mode, fn in fmts.items():
        ids = {fn(b, a) for b, a in zip(pat.fb, pat.fa)} | {fn(b, a) for b, a in zip(pat.tb, pat.ta)}
        ov = len(ds_ids & ids) / max(len(ids), 1)
        if ov > best_ov:
            best_mode, best_ov = mode, ov
    fn = fmts[best_mode]
    log(f"  patterns: account-id format '{best_mode}' fits best ({best_ov:.0%} overlap)")
    pat["src"] = [fn(b, a) for b, a in zip(pat.fb, pat.fa)]
    pat["dst"] = [fn(b, a) for b, a in zip(pat.tb, pat.ta)]
    left = pd.DataFrame({"row": d["pos_idx"], "ts": d["ts_abs"][d["pos_idx"]], "src": d["src_raw_pos"],
                         "dst": d["dst_raw_pos"], "amt": d["amt"][d["pos_idx"]]})

    def match(offset):
        pp = pat.assign(ts=pat["ts"] + offset)
        m = left.merge(pp, on=["ts", "src", "dst"])
        ok = np.isclose(m.amt, m.amt_recv, rtol=1e-3, atol=0.01) | np.isclose(m.amt, m.amt_paid, rtol=1e-3, atol=0.01)
        return m[ok].drop_duplicates("row")

    m, off = match(0), 0
    if len(m) < 0.5 * len(pat):
        cands = [(len(mm), o, mm) for o in range(-14 * 3600, 14 * 3600 + 1, 1800) if o != 0
                 for mm in [match(o)]]
        n_best, off_best, m_best = max(cands, key=lambda t: t[0])
        if n_best > len(m):
            m, off = m_best, off_best
            log(f"  patterns: timestamps shifted by {off / 3600:+.1f}h; corrected")
    rate = len(m) / max(len(pat), 1)
    log(f"  typology match: {len(m):,}/{len(pat):,} patterns found ({rate:.0%})")
    if rate < 0.10:
        log("  typology report SKIPPED (too few matches)")
        return None
    typ = np.full(d["n"], "", dtype=object)
    typ[m["row"].to_numpy()] = m["typology"].to_numpy()
    return typ


# ========================================================================================
# 5. GNN STAGE
# ========================================================================================
def setup_gnn_deps():
    import subprocess
    import sys
    try:
        import torch
    except ImportError:
        log("PyTorch not installed - GNN stage unavailable")
        return False
    ver = torch.__version__.split("+")[0]
    cu = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
    url = f"https://data.pyg.org/whl/torch-{ver}+{cu}.html"
    imported_before = "torch_geometric" in sys.modules

    def pip(*args):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
        if r.returncode != 0:
            log("  pip failed: " + (r.stderr.strip().splitlines() or ["unknown error"])[-1])
        return r.returncode == 0

    try:
        import pyg_lib
    except Exception:
        log(f"Installing pyg-lib for torch {ver}+{cu} ...")
        pip("pyg-lib", "-f", url)
    if not imported_before:
        try:
            import torch_geometric
        except ImportError:
            pip("torch_geometric")

    def pyg_sees_pyg_lib():
        try:
            import torch_geometric.typing as T
            return bool(getattr(T, "WITH_PYG_LIB", False)) and bool(getattr(T, "WITH_EDGE_TIME_NEIGHBOR_SAMPLE", True))
        except Exception:
            return False

    if pyg_sees_pyg_lib():
        return True
    if imported_before:
        log("  PyG imported earlier without pyg-lib — restart runtime, rerun only the pipeline cell.")
    else:
        log(f"  PyG cannot use pyg-lib on torch {ver}+{cu}. GNN stage skipped.")
    return False


def run_gnn_stage(X, d, tr_end, va_end, fit_start):
    import torch
    import torch.nn as nn
    import torch.nn.functional as Fn
    from torch_geometric.data import Data
    from torch_geometric.loader import LinkNeighborLoader
    from torch_geometric.nn import TransformerConv

    n, N, y = d["n"], d["N"], d["y"]
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    log(f"GNN stage on {dev}")
    if not np.isfinite(X).all():
        raise ValueError("non-finite values in feature matrix")

    X_t = torch.from_numpy(X)
    src_t, dst_t = torch.from_numpy(d["src"]), torch.from_numpy(d["dst"])
    ts_t, y_t = torch.from_numpy(d["ts"]), torch.from_numpy(y)
    edge_attr = torch.cat([torch.cat([X_t, torch.zeros(n, 1)], 1),
                           torch.cat([X_t, torch.ones(n, 1)], 1)], 0)
    edge_index = torch.cat([torch.stack([src_t, dst_t]), torch.stack([dst_t, src_t])], 1)
    data = Data(x=torch.ones(N, 1), edge_index=edge_index, edge_attr=edge_attr,
                time=torch.cat([ts_t, ts_t]), num_nodes=N)

    class EdgeGNN(nn.Module):
        def __init__(self, edge_dim, own_dim):
            super().__init__()
            self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
            for i in range(LAYERS):
                self.convs.append(TransformerConv(3 if i == 0 else HIDDEN, HIDDEN, edge_dim=edge_dim))
                self.bns.append(nn.BatchNorm1d(HIDDEN))
            self.own = nn.Sequential(nn.Linear(own_dim, HIDDEN), nn.ReLU())
            self.head = nn.Sequential(nn.Linear(3 * HIDDEN, HIDDEN), nn.ReLU(),
                                      nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 1))

        def forward(self, x, ei, ea, seed, own):
            h = x
            for i, (conv, bn) in enumerate(zip(self.convs, self.bns)):
                h2 = Fn.dropout(Fn.relu(bn(conv(h, ei, ea))), p=DROPOUT, training=self.training)
                h = h2 + h if i > 0 else h2
            z = torch.cat([h[seed[0]], h[seed[1]], self.own(own)], dim=-1)
            return self.head(z).squeeze(-1)

    def make_loader(rows, bs):
        rows_t = torch.as_tensor(rows, dtype=torch.long)
        loader = LinkNeighborLoader(
            data, num_neighbors=NUM_NEIGHBORS, time_attr="time",
            edge_label_index=torch.stack([src_t[rows_t], dst_t[rows_t]]),
            edge_label=y_t[rows_t].float(),
            edge_label_time=ts_t[rows_t] - 1,
            batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
        return loader, rows_t

    checked = {"done": False}

    def run_batch(model, batch, rows_t):
        batch = batch.to(dev)
        g = rows_t[batch.input_id.cpu()]
        x = torch.zeros(batch.x.size(0), 3, device=dev)
        x[:, 0] = 1
        if USE_EGO_MARKERS:
            x[batch.edge_label_index[0], 1] = 1
            x[batch.edge_label_index[1], 2] = 1
        logits = model(x, batch.edge_index, batch.edge_attr, batch.edge_label_index, X_t[g].to(dev))
        if not checked["done"]:
            assert torch.equal(batch.edge_label.long().cpu(), y_t[g]), "seed/label misalignment"
            bt = getattr(batch, "time", None)
            if bt is not None:
                assert int(bt.max()) <= int(ts_t[g].max()) - 1, "sampler returned future edges"
            checked["done"] = True
            log("  sanity checks passed (seed alignment, temporal sampling)")
        return logits, batch

    @torch.no_grad()
    def predict(model, rows):
        model.eval()
        out = np.zeros(len(rows), dtype=np.float32)
        loader, rows_t = make_loader(rows, BATCH_EVAL)
        for batch in loader:
            logits, b = run_batch(model, batch, rows_t)
            out[b.input_id.cpu().numpy()] = logits.float().cpu().numpy()
        return out

    def train(fit_rows, tag, monitor_rows=None):
        pos, neg = fit_rows[y[fit_rows] == 1], fit_rows[y[fit_rows] == 0]
        if len(pos) < 20:
            raise RuntimeError(f"{tag}: only {len(pos)} positives to train on")
        log(f"  [{tag}] fit rows {len(fit_rows):,} | positives {len(pos):,}")
        model = EdgeGNN(X.shape[1] + 1, X.shape[1]).to(dev)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=dev))
        best = {"ap": -1.0, "ep": 0, "state": None}
        for ep in range(1, EPOCHS + 1):
            model.train()
            k = min(len(neg), NEG_RATIO * len(pos))
            seeds = np.concatenate([pos, rng.choice(neg, k, replace=False)])
            rng.shuffle(seeds)
            loader, rows_t = make_loader(seeds, BATCH_TRAIN)
            tot, nb = 0.0, 0
            for batch in loader:
                opt.zero_grad()
                logits, b = run_batch(model, batch, rows_t)
                loss = crit(logits, b.edge_label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                tot, nb = tot + loss.item(), nb + 1
            sched.step()
            if ep % 5 == 0 or ep == 1:
                log(f"  [{tag}] epoch {ep:02d}/{EPOCHS}  loss {tot / max(nb, 1):.4f}")
            if monitor_rows is not None and (ep % EVAL_EVERY == 0 or ep == EPOCHS):
                ap = average_precision_score(y[monitor_rows], predict(model, monitor_rows))
                log(f"  [{tag}] epoch {ep:02d} validation-proxy PR-AUC {ap:.4f}")
                if ap > best["ap"]:
                    best.update(ap=ap, ep=ep, state=copy.deepcopy(model.state_dict()))
        if best["state"] is not None:
            model.load_state_dict(best["state"])
            log(f"  [{tag}] restored best epoch {best['ep']} (proxy PR-AUC {best['ap']:.4f})")
        return model

    out = {"final": np.full(n, np.nan, dtype=np.float32), "oof": None, "s0": None}
    if STACK_OOF:
        out["oof"] = np.full(n, np.nan, dtype=np.float32)
        bounds = [int(b * n) for b in GNN_FOLD_BOUNDS] + [tr_end]
        out["s0"] = bounds[0]
        for k in range(len(bounds) - 1):
            model = train(np.arange(0, bounds[k]), f"fold{k + 1}")
            rows = np.arange(bounds[k], bounds[k + 1])
            out["oof"][rows] = predict(model, rows)
            if y[rows].sum():
                log(f"  [fold{k + 1}] standalone OOF PR-AUC {average_precision_score(y[rows], out['oof'][rows]):.4f}")
            del model

    val_rows = np.arange(tr_end, va_end)
    vpos = val_rows[y[val_rows] == 1]
    vneg = rng.choice(val_rows[y[val_rows] == 0], min(100_000, int((y[val_rows] == 0).sum())), replace=False)
    monitor = np.sort(np.concatenate([vpos, vneg]))
    model = train(np.arange(fit_start, tr_end), "final", monitor_rows=monitor)
    rows = np.arange(tr_end, n)
    out["final"][rows] = predict(model, rows)
    log(f"  [final] standalone GNN PR-AUC  val {average_precision_score(y[tr_end:va_end], out['final'][tr_end:va_end]):.4f}"
        f"  test {average_precision_score(y[va_end:], out['final'][va_end:]):.4f}")
    return out


# ========================================================================================
# 6. REPORTING
# ========================================================================================
def print_importance(run, names, k=20):
    imp = run["model"].get_booster().get_score(importance_type="gain")
    print(f"\nTop features by GAIN ({run['name']}):")
    for key, v in sorted(imp.items(), key=lambda kv: -kv[1])[:k]:
        idx = int(key[1:]) if key.startswith("f") and key[1:].isdigit() else None
        print(f"  {names[idx] if idx is not None and idx < len(names) else key:<28} {v:.1f}")


def diagnostics(res, y_va, y_te):
    p_te, p_va, thr = res["p_te"], res["p_va"], res["thr"]
    pred = (p_te >= thr).astype(int)
    print("\n" + "=" * 80 + f"\nHEADLINE (best validation PR-AUC): {res['name']}\n" + "=" * 80)
    print(classification_report(y_te, pred, target_names=["Licit", "Illicit"], digits=4, zero_division=0))
    lo, hi = bootstrap_ap(y_te, p_te, N_BOOT)
    blo, bhi = bootstrap_ap(y_te, p_te, N_BOOT, block=max(len(y_te) // 100, 1))
    base = y_te.mean()
    print(f"Test PR-AUC {res['ap_test']:.4f}  (95% CI iid {lo:.4f}-{hi:.4f}; time-block {blo:.4f}-{bhi:.4f}; "
          f"base rate {base:.5f}; lift {res['ap_test'] / base:.0f}x)")
    print(f"Val  PR-AUC {res['ap_val']:.4f}   threshold (picked on val, beta={THRESH_BETA}) = {thr:.5f}")
    print(f"Alerts on test: {int(pred.sum()):,} of {len(pred):,} ({pred.mean() * 100:.3f}%)")
    order = np.argsort(-p_te)
    print("\nAlert-budget view:")
    for fr in ALERT_BUDGETS:
        k = max(int(fr * len(y_te)), 1)
        tp = int(y_te[order[:k]].sum())
        print(f"  top {fr * 100:.1f}% ({k:,} alerts): precision {tp / k:.3f}  recall {tp / max(y_te.sum(), 1):.3f}")
    print("\nDrift check - PR-AUC over time slices:")
    for nm, y_, p_, parts in (("val", y_va, p_va, 2), ("test", y_te, p_te, 4)):
        out = []
        for chunk in np.array_split(np.arange(len(y_)), parts):
            ap_ = f"{average_precision_score(y_[chunk], p_[chunk]):.3f}" if y_[chunk].sum() else "n/a"
            out.append(f"{ap_} (pos {int(y_[chunk].sum())}, {y_[chunk].mean() * 100:.2f}%)")
        print(f"  {nm}: " + "   ".join(out))


def typology_table(runs, y_te, typ_te):
    pos = y_te == 1
    names = sorted(set(typ_te[pos]))
    k = max(int(0.01 * len(y_te)), 1)
    tops = {r["short"]: np.isin(np.arange(len(y_te)), np.argsort(-r["p_te"])[:k]) for r in runs}
    preds = {r["short"]: r["p_te"] >= r["thr"] for r in runs}
    head = "".join(f"{r['short'][:9]:>11}" for r in runs)
    print("\nRecall by laundering typology on TEST   [recall at validation threshold | recall in top 1% alerts]")
    print(f"  {'typology':<22}{'n':>5}{head}")
    for t in names:
        m = pos & (typ_te == t)
        cells = "".join(f"  {preds[r['short']][m].mean():.2f}|{tops[r['short']][m].mean():.2f}" for r in runs)
        print(f"  {(t or '(not in patterns file)'):<22}{int(m.sum()):>5}{cells}")


# ========================================================================================
# 7. SYNTHETIC DATA (SMOKE only)
# ========================================================================================
def make_synthetic(path, patterns_path, n_bg=250_000, n_struct=90, seed=0):
    rng = np.random.default_rng(seed)
    n_acc = 6000
    w = 1.0 / np.arange(1, n_acc + 1) ** 0.8
    w /= w.sum()
    t0 = pd.Timestamp("2022-09-01")
    fmts = np.array(["Reinvestment", "Cheque", "Credit Card", "ACH", "Cash", "Wire", "Bitcoin"])
    fp = np.array([.15, .2, .2, .25, .1, .08, .02])
    cur = np.array(["US Dollar", "Euro", "Yen", "Bitcoin"])
    s, r = rng.choice(n_acc, n_bg, p=w), rng.choice(n_acc, n_bg, p=w)
    minutes = np.sort(rng.integers(0, 10 * 1440, n_bg))
    amt = rng.lognormal(7, 1.5, n_bg)
    fm = rng.choice(fmts, n_bg, p=fp)
    cross = rng.random(n_bg) < 0.12
    bank = lambda a: f"{(a % 40):03d}"
    acct = lambda a: f"8{a:07X}"
    rows = []
    for i in range(n_bg):
        c1 = cur[rng.integers(0, 3)]
        c2 = cur[rng.integers(0, 3)] if cross[i] else c1
        rows.append((t0 + pd.Timedelta(minutes=int(minutes[i])), bank(s[i]), acct(s[i]), bank(r[i]), acct(r[i]),
                     amt[i] * (1.1 if cross[i] else 1.0), c2, amt[i], c1, fm[i], 0))
    pats, nxt = [], n_acc
    for k in range(n_struct):
        hub, sink = int(rng.integers(0, n_acc)), int(rng.integers(0, n_acc))
        start, tot = int(rng.integers(0, 9 * 1440)), float(rng.lognormal(10, 1))
        edges = []
        for m in range(6):
            mule, a = nxt, tot / 6 * rng.uniform(0.95, 1.05)
            nxt += 1
            t1 = start + int(rng.integers(0, 10))
            edges.append((t1, hub, mule, a))
            edges.append((t1 + int(rng.integers(5, 120)), mule, sink, a * 0.98))
        for (tm, a1, a2, a) in edges:
            row = (t0 + pd.Timedelta(minutes=tm), bank(a1), acct(a1), bank(a2), acct(a2), a, "US Dollar", a,
                   "US Dollar", "ACH", 1)
            rows.append(row)
            pats.append((k, row))
    df = pd.DataFrame(rows, columns=RAW_NAMES).sample(frac=1.0, random_state=seed)
    df["Timestamp"] = df["Timestamp"].dt.strftime("%Y/%m/%d %H:%M")
    df.to_csv(path, index=False)
    with open(patterns_path, "w") as fh:
        for k in sorted({a for a, _ in pats}):
            fh.write("BEGIN LAUNDERING ATTEMPT - SCATTER-GATHER:  Max 2 hops\n")
            for a, row in pats:
                if a == k:
                    fh.write(",".join([row[0].strftime("%Y/%m/%d %H:%M"), row[1], row[2], row[3], row[4],
                                       f"{row[5]:.2f}", row[6], f"{row[7]:.2f}", row[8], row[9], "1"]) + "\n")
            fh.write("END LAUNDERING ATTEMPT - SCATTER-GATHER\n")
    log(f"Synthetic data: {len(df):,} rows, {int(df['Is Laundering'].sum()):,} laundering")


# ========================================================================================
# 8. MAIN
# ========================================================================================
_FOUND = {"orig": None}


def _localize(p):
    _FOUND["orig"] = p
    if COPY_TO_LOCAL and p.startswith(os.path.dirname(DRIVE_ROOT) + "/"):
        dst = os.path.join(LOCAL_DIR, os.path.basename(p))
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(p):
            t0 = time.time()
            log(f"  copying {os.path.basename(p)} ({os.path.getsize(p) / 1e6:.0f} MB) from Drive ...")
            shutil.copyfile(p, dst)
            log(f"  copied in {time.time() - t0:.0f}s")
        return dst
    return p


def find_or_upload(path, required=True):
    if path is None:
        return None
    base = os.path.basename(path)
    for cand in (path, os.path.join(DRIVE_DIR, base), os.path.join(DRIVE_ROOT, base)):
        if os.path.exists(cand):
            return _localize(cand)
    root = DRIVE_ROOT
    if os.path.isdir(root):
        for pat in (f"{root}/*/{base}", f"{root}/*/*/{base}"):
            hits = sorted(glob.glob(pat))
            if hits:
                log(f"  found {base} at {hits[0]}")
                return _localize(hits[0])
    local = os.path.join(LOCAL_DIR, base)
    if os.path.exists(local):
        return _localize(local)
    if not required:
        log(f"'{base}' not found - skipping (optional)")
        return None
    try:
        from google.colab import files
        log(f"'{base}' not found in Drive - choose the file to upload ...")
        up = files.upload()
        if not up:
            raise FileNotFoundError(f"no file uploaded for {path}")
        return next(iter(up))
    except ImportError:
        raise FileNotFoundError(f"file not found: {path}")


def default_output_dir():
    orig = _FOUND["orig"]
    if orig and orig.startswith(os.path.dirname(DRIVE_ROOT) + "/"):
        return os.path.join(os.path.dirname(orig), "aml_outputs")
    if os.path.isdir(DRIVE_DIR):
        return os.path.join(DRIVE_DIR, "aml_outputs")
    return os.path.join(LOCAL_DIR, "aml_outputs")


def cache_key_for(path, d):
    sig = (f"{FEATURE_VERSION}|{WINDOWS}|{DISTINCT_OWN}|{DISTINCT_CROSS_WINDOW}|{TRAIN_FRAC}|{CAT_COLS}|"
           f"{int(bool(d.get('created_ok')))}|{d['n']}|{d['N']}|{int(d['y'].sum())}|{int(d['ts_abs'][0])}|"
           f"{int(d['ts_abs'][-1])}")
    return f"{os.path.splitext(os.path.basename(path))[0]}-{d['n']}-{hashlib.md5(sig.encode()).hexdigest()[:10]}"


def is_fingerprint(nm):
    return nm.startswith(("payment_format=", "currency_paid=")) or nm in (
        "same_currency", "amt_equal", "amt_logratio", "same_bank")


def run_experiment(tag, X_full, names_full, cols, d, burn_end, tr_end, va_end, typ, out_dir, use_gnn, cache_key=""):
    os.makedirs(out_dir, exist_ok=True)
    n, y = d["n"], d["y"]
    names = [names_full[i] for i in cols]
    X = X_full[:, cols]
    mu, sd = X[burn_end:tr_end].mean(0), X[burn_end:tr_end].std(0) + 1e-6
    X -= mu
    X /= sd
    ytr, yva, yte = y[:tr_end], y[tr_end:va_end], y[va_end:]
    Xva, Xte = X[tr_end:va_end], X[va_end:]
    print("\n" + "#" * 80 + f"\n# EXPERIMENT '{tag}': {X.shape[1]} features, GNN {'ON' if use_gnn else 'off'}\n" + "#" * 80)

    log(f"[{tag}] tabular models")
    runs = [fit_xgb("tabular (all train rows)", "tab", X[:tr_end], ytr, Xva, yva, Xte, yte)]
    if burn_end >= int(0.01 * n) and y[burn_end:tr_end].sum() >= 300:
        runs.append(fit_xgb("tabular (burn-in applied)", "tab-burn", X[burn_end:tr_end], y[burn_end:tr_end],
                            Xva, yva, Xte, yte))
    best_tab = max(runs, key=lambda r: r["ap_val"])

    audit = None
    if AUDIT_FINGERPRINT_ONLY:
        fpc = [i for i, nm in enumerate(names) if is_fingerprint(nm)]
        if fpc:
            audit = fit_xgb("AUDIT: fingerprint columns only", "audit", X[:tr_end][:, fpc], ytr, Xva[:, fpc], yva,
                            Xte[:, fpc], yte)

    gnn = None
    gpath = None
    if use_gnn and CACHE_GNN and not STACK_OOF:
        hp = hashlib.md5(repr((NUM_NEIGHBORS, HIDDEN, LAYERS, DROPOUT, EPOCHS, LR, EVAL_EVERY, NEG_RATIO, POS_WEIGHT,
                               BATCH_TRAIN, USE_EGO_MARKERS, GNN_BURN_IN, burn_end, SEED, tag)).encode()).hexdigest()[:8]
        gpath = os.path.join(OUTPUT_DIR, "gnn_cache", f"{cache_key}-{tag}-{hp}.npy")
    if gpath and os.path.exists(gpath) and not FORCE_RECOMPUTE:
        gnn = dict(final=np.load(gpath), oof=None, s0=None)
        log(f"[{tag}] GNN scores loaded from {gpath} (delete it or set FORCE_RECOMPUTE to retrain)")
    elif use_gnn and setup_gnn_deps():
        log(f"[{tag}] GNN stage")
        try:
            gnn = run_gnn_stage(X, d, tr_end, va_end, burn_end if GNN_BURN_IN else 0)
            if gpath and gnn is not None:
                os.makedirs(os.path.dirname(gpath), exist_ok=True)
                np.save(gpath, gnn["final"])
                log(f"[{tag}] GNN scores saved to {gpath}")
        except ImportError as e:
            log(f"GNN stage skipped: {e}")
    blend = None
    if gnn is not None:
        g = gnn["final"]
        runs.append(make_run("GNN alone", "gnn", g[tr_end:va_end], g[va_end:], yva, yte))
        blend = blend_run(best_tab, g[tr_end:va_end], g[va_end:], yva, yte)
        runs.append(blend)
        if gnn["oof"] is not None:
            s0 = gnn["s0"]
            Xs = np.hstack([X[s0:], gnn["oof"][s0:, None]])
            a, b = tr_end - s0, va_end - s0
            Xs[a:, -1] = g[tr_end:]
            runs.append(fit_xgb("stacked OOF (tabular + GNN score)", "stack", Xs[:a], y[s0:tr_end], Xs[a:b], yva,
                                Xs[b:], yte))

    print(f"\n[{tag}] results (headline = best VALIDATION PR-AUC)")
    print("{:<42} {:>8} {:>8} {:>7} {:>7} {:>7}".format("run", "val AP", "test AP", "prec", "recall", "F1"))
    for r in runs + ([audit] if audit else []):
        print("{:<42} {:>8.4f} {:>8.4f} {:>7.3f} {:>7.3f} {:>7.3f}".format(
            r["name"], r["ap_val"], r["ap_test"], r["precision"], r["recall"], r["f1"]))
    head = max(runs, key=lambda r: r["ap_val"])
    blk = max(len(yte) // 100, 1)

    gnn_gain = None
    if blend is not None:
        gnn_run = [r for r in runs if r["short"] == "gnn"][0]
        print(f"\n[{tag}] WHAT DOES THE GNN ADD?  (paired bootstrap on test PR-AUC, {N_BOOT} resamples)")
        for label, a_, b_ in (("blend minus best tabular", blend, best_tab), ("GNN alone minus best tabular", gnn_run, best_tab)):
            m_, (lo_, hi_), pg = paired_bootstrap_ap(yte, a_["p_te"], b_["p_te"], N_BOOT, block=blk)
            print(f"  {label:<32} diff {m_:+.4f}  95% time-block CI [{lo_:+.4f}, {hi_:+.4f}]  P(>0) {pg:.2f}")
        print(f"  blend weights (tabular, GNN): {blend['weights'][0]:+.2f}, {blend['weights'][1]:+.2f}")
        gnn_gain = dict(blend_minus_tab=float(blend["ap_test"] - best_tab["ap_test"]),
                        gnn_alone_minus_tab=float(gnn_run["ap_test"] - best_tab["ap_test"]))

    print(f"\n[{tag}] paired bootstrap: headline minus each other run")
    for r in runs:
        if r is head:
            continue
        for lab, bk in (("iid", 0), ("time-block", blk)):
            m_, (lo_, hi_), pg = paired_bootstrap_ap(yte, head["p_te"], r["p_te"], N_BOOT, block=bk)
            print(f"  vs {r['name']:<40} [{lab:<10}] diff {m_:+.4f}  95% CI [{lo_:+.4f}, {hi_:+.4f}]  P {pg:.2f}")

    diagnostics(head, yva, yte)
    print_importance(best_tab, names)
    if typ is not None:
        typology_table(runs, yte, typ[va_end:])

    summary = {r["name"]: {k: v for k, v in r.items() if k not in ("p_va", "p_te", "model")} for r in runs}
    if audit:
        summary[audit["name"]] = {k: v for k, v in audit.items() if k not in ("p_va", "p_te", "model")}
    summary.update(headline=head["name"], features=names, gnn_gain=gnn_gain)
    with open(os.path.join(out_dir, "results.json"), "w") as fh:
        json.dump(summary, fh, indent=2, default=float)
    best_tab["model"].save_model(os.path.join(out_dir, "xgb_tabular.json"))
    np.savez_compressed(os.path.join(out_dir, "all_runs_test_scores.npz"), y_test=yte,
                        **{r["short"].replace("-", "_"): r["p_te"] for r in runs})
    return dict(tag=tag, head=head["name"], ap_val=head["ap_val"], ap_test=head["ap_test"], precision=head["precision"],
                recall=head["recall"], f1=head["f1"], gnn_gain=gnn_gain,
                best_tab=(best_tab["name"], best_tab["ap_val"], best_tab["ap_test"]),
                audit=(audit["ap_val"], audit["ap_test"]) if audit else None)


def main():
    global CSV_PATH, PATTERNS_PATH, OUTPUT_DIR, N_BOOT, USE_GNN, SMOKE, STACK_OOF, FEATURE_SETS
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default=_CFG["csv"])
    ap.add_argument("--patterns", default=_CFG["patterns"])
    ap.add_argument("--out", default=None)
    ap.add_argument("--sets", default=",".join(FEATURE_SETS), help="comma list of: structural, full")
    ap.add_argument("--no-gnn", action="store_true")
    ap.add_argument("--stack", action="store_true")
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--boot", type=int, default=N_BOOT)
    args, _ = ap.parse_known_args()
    CSV_PATH, PATTERNS_PATH, N_BOOT = args.csv, args.patterns, args.boot
    OUTPUT_DIR = args.out if args.out else _CFG["out"]
    FEATURE_SETS = [t.strip() for t in args.sets.split(",") if t.strip()]
    USE_GNN, SMOKE, STACK_OOF = USE_GNN and not args.no_gnn, SMOKE or args.smoke, STACK_OOF or args.stack

    # ★ CHANGE 3: skip-if-complete. If a full run is already on Drive, we're done.
    #             Costs ~2 s. Set FORCE_RECOMPUTE=True to override.
    if (not SMOKE) and (not FORCE_RECOMPUTE) and OUTPUT_DIR \
            and os.path.exists(os.path.join(OUTPUT_DIR, "summary.json")):
        log(f"✅ Detection already complete at {OUTPUT_DIR} — skipping. "
            f"(Set FORCE_RECOMPUTE=True or delete summary.json to re-run.)")
        try:
            os.makedirs(STATE_DIR, exist_ok=True)
            open(os.path.join(STATE_DIR, "pipeline_done.flag"), "w").write("1")
        except OSError:
            pass
        return

    np.random.seed(SEED)
    t_start = time.time()

    if SMOKE:
        OUTPUT_DIR = OUTPUT_DIR or ("/content/aml_smoke" if os.path.isdir("/content") else "aml_smoke")
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        CSV_PATH = os.path.join(OUTPUT_DIR, "synthetic_trans.csv")
        PATTERNS_PATH = os.path.join(OUTPUT_DIR, "synthetic_patterns.txt")
        make_synthetic(CSV_PATH, PATTERNS_PATH)
    else:
        if MOUNT_DRIVE and not os.path.isdir(DRIVE_ROOT):
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except ImportError:
                pass
        CSV_PATH = find_or_upload(CSV_PATH)
        if OUTPUT_DIR is None:
            OUTPUT_DIR = default_output_dir()
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        PATTERNS_PATH = find_or_upload(PATTERNS_PATH, required=False)

    log(f"outputs and GNN-score cache: {OUTPUT_DIR}")
    log(f"numba: {'available' if HAVE_NUMBA else 'NOT available'}")
    log("1/4 Loading + chronological sort")
    d = load_data(CSV_PATH)
    n, y = d["n"], d["y"]
    tr_end, va_end = int(TRAIN_FRAC * n), int((TRAIN_FRAC + VAL_FRAC) * n)
    log(f"  {n:,} edges, {d['N']:,} accounts | train {tr_end:,}  val {va_end - tr_end:,}  test {n - va_end:,}")
    for nm, sl in (("train", slice(0, tr_end)), ("val", slice(tr_end, va_end)), ("test", slice(va_end, n))):
        log(f"  {nm}: illicit {int(y[sl].sum()):,} ({y[sl].mean() * 100:.3f}%)")
    log(f"  time range: {pd.Timestamp(int(d['ts_abs'].min()), unit='s')} -> "
        f"{pd.Timestamp(int(d['ts_abs'].max()), unit='s')}   (full HI-Small: 5,078,345 edges, 5,177 laundering)")

    d["created_ok"] = audit_created(d)
    ckey = cache_key_for(CSV_PATH, d)
    fdir = os.path.join(CACHE_DIR, ckey)
    fx, fn, fc = (os.path.join(fdir, f) for f in ("X.npy", "names.json", "cold.npy"))
    log("2/4 Causal features")
    if CACHE_FEATURES and not FORCE_RECOMPUTE and all(os.path.exists(f) for f in (fx, fn, fc)):
        X, names, cold = np.load(fx), json.load(open(fn)), np.load(fc)
        log(f"  loaded from cache {fdir}")
    else:
        X, names, extras = causal_features(d)
        cold = extras["s_new"] | extras["r_new"]
        del extras
        if CACHE_FEATURES:
            try:
                os.makedirs(fdir, exist_ok=True)
                np.save(fx, X)
                json.dump(names, open(fn, "w"))
                np.save(fc, cold)
                log(f"  features cached to {fdir}")
            except OSError as e:
                log(f"  feature cache write failed ({e}); continuing without persistence")
    log(f"  {X.shape[1]} features")
    burn_end, rates = choose_burn_in(cold, tr_end, n)
    log("  cold-start (unseen sender or receiver) per 1% of timeline, train: "
        + " ".join(f"{r:.0%}" for r in rates[:12]) + " ...")
    log(f"  cold-start val {cold[tr_end:va_end].mean():.2%} | test {cold[va_end:].mean():.2%}")
    log(f"  burn-in: first {burn_end:,} rows ({burn_end / max(tr_end, 1):.0%} of train); positives set aside: {int(y[:burn_end].sum()):,}")
    typ = load_typologies(PATTERNS_PATH, d) if PATTERNS_PATH else None
    del d["df"]

    fp = [is_fingerprint(nm) for nm in names]
    all_cols = list(range(len(names)))
    sets = {"structural": [i for i, f_ in enumerate(fp) if not f_], "full": all_cols}
    log(f"  structural set: {len(sets['structural'])} features | full set: {len(all_cols)} (fingerprint cols: {sum(fp)})")
    if not any(fp):
        log("  no fingerprint fields -> 'structural' and 'full' identical; running once")
        sets = {"structural": all_cols}

    log("3/4 Experiments")
    results = {}
    for tag in FEATURE_SETS:
        if tag in sets:
            results[tag] = run_experiment(tag, X, names, sets[tag], d, burn_end, tr_end, va_end, typ,
                                          os.path.join(OUTPUT_DIR, tag), USE_GNN and tag in GNN_FEATURE_SETS, ckey)

    log("4/4 Summary")
    print("\n" + "=" * 96 + "\nSUMMARY (headline per experiment is chosen on VALIDATION PR-AUC)\n" + "=" * 96)
    print("{:<11} {:<40} {:>8} {:>8} {:>6} {:>6} {:>6}".format("features", "headline run", "val AP", "test AP", "prec", "rec", "F1"))
    for tag, r in results.items():
        print("{:<11} {:<40} {:>8.4f} {:>8.4f} {:>6.3f} {:>6.3f} {:>6.3f}".format(
            tag, r["head"][:40], r["ap_val"], r["ap_test"], r["precision"], r["recall"], r["f1"]))
    for tag, r in results.items():
        if r["audit"]:
            print(f"  [{tag}] AUDIT - fingerprints ALONE: val AP {r['audit'][0]:.4f} / test AP {r['audit'][1]:.4f}")
        if r["gnn_gain"]:
            g_ = r["gnn_gain"]
            print(f"  [{tag}] GNN contribution: blend {g_['blend_minus_tab']:+.4f} vs best tab, "
                  f"GNN alone {g_['gnn_alone_minus_tab']:+.4f}")
    if INPUT_KIND == "neo4j":
        print("  reference - your earlier Neo4j pipeline: val AP 0.4929 | test AP 0.5426 | F1 0.549")
    with open(os.path.join(OUTPUT_DIR, "summary.json"), "w") as fh:
        json.dump(results, fh, indent=2, default=float)

    # ★ CHANGE 4: mark complete so Cell 5 (and future re-runs) can act on it.
    try:
        os.makedirs(STATE_DIR, exist_ok=True)
        open(os.path.join(STATE_DIR, "pipeline_done.flag"), "w").write("1")
    except OSError:
        pass

    log(f"Done in {(time.time() - t_start) / 60:.1f} min. Outputs in {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

[13:34:09]   copying HI-Small_Trans.csv (476 MB) from Drive ...
[13:34:10]   copied in 1s
[13:34:10]   copying HI-Small_Patterns.txt (0 MB) from Drive ...
[13:34:10]   copied in 0s
[13:34:10] outputs and GNN-score cache: /content/drive/MyDrive/aml/aml_outputs
[13:34:10] numba: available
[13:34:10] 1/4 Loading + chronological sort
[13:34:22]   raw IBM file detected: payment format, currencies, banks and amounts are used
[13:34:29]   5,078,345 edges, 515,088 accounts | train 3,047,007  val 1,015,669  test 1,015,669
[13:34:29]   train: illicit 2,298 (0.075%)
[13:34:29]   val: illicit 1,082 (0.107%)
[13:34:29]   test: illicit 1,797 (0.177%)
[13:34:29]   time range: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00   (full HI-Small: 5,078,345 edges, 5,177 laundering)
[13:34:29] 2/4 Causal features
[13:37:00]   features cached to /content/drive/MyDrive/aml/feature_cache/HI-Small_Trans-5078345-ddab4fb25c
[13:37:00]   106 features
[13:37:00]   cold-start (unseen sender or receiver) per 1% of timeline

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [13:37:30] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[13:37:33] XGBoost [tabular (burn-in applied)]  train=1,980,555 (pos 1,987)  device=cuda
[0]	validation_0-aucpr:0.32794
[200]	validation_0-aucpr:0.55134
[400]	validation_0-aucpr:0.55735
[600]	validation_0-aucpr:0.56283
[800]	validation_0-aucpr:0.56314
[810]	validation_0-aucpr:0.56328
[13:38:02] [structural] GNN stage
[13:38:02] GNN stage on cuda
[13:38:04]   [final] fit rows 3,047,007 | positives 2,298
[13:38:06]   sanity checks passed (seed alignment, temporal sampling)
[13:38:13]   [final] epoch 01/60  loss 0.4051
[13:38:48]   [final] epoch 05/60  loss 0.1559
[13:39:30]   [final] epoch 10/60  loss 0.1394
[13:39:53]   [final] epoch 10 validation-proxy PR-AUC 0.6861
[13:40:35]   [final] epoch 15/60  loss 0.1320
[13:41:18]   [final] epoch 20/60  loss 0.1194
[13:41:41]   [final] epoch 20 validation-proxy PR-AUC 0.7290
[13:42:23]   [final] epoch 25/60  loss 0.1105
[13:43:05]   [final] epoch 30/60  loss 0.1053
[13:43:28]   [final] epoch 30 validation-proxy PR-AUC 0.7274
[13:44:10]   [final

In [ ]:
import os, subprocess

items = {
    "Neo4j data backup":   "/content/drive/MyDrive/aml/neo4j_backup.tar.gz",
    "Neo4j tarball cache": "/content/drive/MyDrive/aml/neo4j-5.20.0-unix.tar.gz",
    "Flag: neo4j_loaded":  "/content/drive/MyDrive/aml/state/neo4j_loaded.flag",
    "Flag: pipeline_done": "/content/drive/MyDrive/aml/state/pipeline_done.flag",
    "Flag: graphrag_done": "/content/drive/MyDrive/aml/state/graphrag_done.flag",
    "Detection summary":   "/content/drive/MyDrive/aml/aml_outputs/summary.json",
    "Structural results":  "/content/drive/MyDrive/aml/aml_outputs/structural/results.json",
    "Explanations":        "/content/drive/MyDrive/aml/aml_outputs/explanations.json",
    "Feature cache dir":   "/content/drive/MyDrive/aml/feature_cache",
}

for label, path in items.items():
    ok = os.path.exists(path)
    size = ""
    if ok and os.path.isfile(path):
        size = f"  ({os.path.getsize(path)/1e6:.1f} MB)"
    elif ok and os.path.isdir(path):
        n = len(os.listdir(path))
        size = f"  ({n} files)"
    print(f"{'✅' if ok else '❌'} {label:<22} {path}{size}")

❌ Neo4j data backup      /content/drive/MyDrive/aml/neo4j_backup.tar.gz
❌ Neo4j tarball cache    /content/drive/MyDrive/aml/neo4j-5.20.0-unix.tar.gz
❌ Flag: neo4j_loaded     /content/drive/MyDrive/aml/state/neo4j_loaded.flag
❌ Flag: pipeline_done    /content/drive/MyDrive/aml/state/pipeline_done.flag
❌ Flag: graphrag_done    /content/drive/MyDrive/aml/state/graphrag_done.flag
❌ Detection summary      /content/drive/MyDrive/aml/aml_outputs/summary.json
❌ Structural results     /content/drive/MyDrive/aml/aml_outputs/structural/results.json
❌ Explanations           /content/drive/MyDrive/aml/aml_outputs/explanations.json
❌ Feature cache dir      /content/drive/MyDrive/aml/feature_cache


In [ ]:
# Step 1 (fixed) — mount Drive FIRST, then check
import os

# Mount Drive if not already mounted
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Now check — these paths should exist
items = {
    "Neo4j data backup":   "/content/drive/MyDrive/aml/neo4j_backup.tar.gz",
    "Neo4j tarball cache": "/content/drive/MyDrive/aml/neo4j-5.20.0-unix.tar.gz",
    "Flag: neo4j_loaded":  "/content/drive/MyDrive/aml/state/neo4j_loaded.flag",
    "Flag: pipeline_done": "/content/drive/MyDrive/aml/state/pipeline_done.flag",
    "Flag: graphrag_done": "/content/drive/MyDrive/aml/state/graphrag_done.flag",
    "Detection summary":   "/content/drive/MyDrive/aml/aml_outputs/summary.json",
    "Structural results":  "/content/drive/MyDrive/aml/aml_outputs/structural/results.json",
    "Explanations":        "/content/drive/MyDrive/aml/aml_outputs/explanations.json",
    "Feature cache dir":   "/content/drive/MyDrive/aml/feature_cache",
    "HI-Small_Trans.csv":  "/content/drive/MyDrive/ibm aml/HI-Small_Trans.csv",
}

for label, path in items.items():
    ok = os.path.exists(path)
    size = ""
    if ok and os.path.isfile(path):
        size = f"  ({os.path.getsize(path)/1e6:.1f} MB)"
    elif ok and os.path.isdir(path):
        n = len(os.listdir(path))
        size = f"  ({n} files)"
    print(f"{'✅' if ok else '❌'} {label:<22} {path}{size}")

Mounted at /content/drive
✅ Neo4j data backup      /content/drive/MyDrive/aml/neo4j_backup.tar.gz  (669.6 MB)
❌ Neo4j tarball cache    /content/drive/MyDrive/aml/neo4j-5.20.0-unix.tar.gz
✅ Flag: neo4j_loaded     /content/drive/MyDrive/aml/state/neo4j_loaded.flag  (0.0 MB)
✅ Flag: pipeline_done    /content/drive/MyDrive/aml/state/pipeline_done.flag  (0.0 MB)
❌ Flag: graphrag_done    /content/drive/MyDrive/aml/state/graphrag_done.flag
✅ Detection summary      /content/drive/MyDrive/aml/aml_outputs/summary.json  (0.0 MB)
✅ Structural results     /content/drive/MyDrive/aml/aml_outputs/structural/results.json  (0.0 MB)
❌ Explanations           /content/drive/MyDrive/aml/aml_outputs/explanations.json
✅ Feature cache dir      /content/drive/MyDrive/aml/feature_cache  (1 files)
✅ HI-Small_Trans.csv     /content/drive/MyDrive/ibm aml/HI-Small_Trans.csv  (475.7 MB)


In [ ]:
# ============================================================
# CELL 5 — Graph RAG explainability (Gemini, self-sufficient)
# ============================================================
import json, os, time, numpy as np, pandas as pd

# ---------- Config ----------
DRIVE_ROOT     = "/content/drive/MyDrive"
AML_DIR        = f"{DRIVE_ROOT}/ibm aml"
RAW_CSV        = f"{AML_DIR}/HI-Small_Trans.csv"
OUTPUT_DIR     = f"{DRIVE_ROOT}/aml/aml_outputs"
STATE_DIR      = f"{DRIVE_ROOT}/aml/state"
FLAG_PIPELINE  = f"{STATE_DIR}/pipeline_done.flag"

NEO4J_URI      = "bolt://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "amlpassword123"

# Gemini model — swap to "gemini-2.5-pro" for max quality
GEMINI_MODEL    = "gemini-3.5-flash-lite"
GEMINI_FALLBACK = ["gemini-1.5-flash", "gemini-2.0-flash-lite"]
N_ALERTS        = 10   # how many top-scoring alerts to explain

# ---------- 0. Sanity ----------
if not os.path.isdir(DRIVE_ROOT):
    from google.colab import drive
    drive.mount("/content/drive")

if not os.path.exists(FLAG_PIPELINE):
    raise RuntimeError("Cell 4 not complete — run Cell 4 first.")

# ---------- 1. Ensure Neo4j driver ----------
from neo4j import GraphDatabase

try:
    driver.verify_connectivity()          # reuse Cell 2's driver if alive
    print("✅ Reusing existing Neo4j driver")
except Exception:
    print("Rebuilding Neo4j driver …")
    driver = GraphDatabase.driver(NEO4J_URI,
                                  auth=(NEO4J_USER, NEO4J_PASSWORD),
                                  keep_alive=True,
                                  max_connection_lifetime=300,
                                  connection_acquisition_timeout=10)
    driver.verify_connectivity()
    print("✅ New driver connected")

def run_cypher(q, **params):
    with driver.session() as s:
        return list(s.run(q, **params))

# ---------- 2. Locate results ----------
subdir = next((c for c in ("structural", "full", "neo4j")
               if os.path.exists(f"{OUTPUT_DIR}/{c}/results.json")), None)
if subdir is None:
    raise FileNotFoundError(f"No results.json under {OUTPUT_DIR}")

r = json.load(open(f"{OUTPUT_DIR}/{subdir}/results.json"))
print(f"Loaded results from '{subdir}': headline = {r['headline']}")

# ---------- 3. Load scores + threshold (correct scale) ----------
z = np.load(f"{OUTPUT_DIR}/{subdir}/all_runs_test_scores.npz")
y_test   = z["y_test"]
best_run = r["headline"]
entry    = r[best_run]
short    = entry.get("short", "tab")
short_key = short.replace("-", "_")

if short_key in z.files:
    p_te = z[short_key]
else:
    p_te = z["tab"] if "tab" in z.files else z[[k for k in z.files if k != "y_test"][0]]
    print(f"⚠️  '{short_key}' not in npz; using fallback. Available: {z.files}")

thr = entry["thr"]
assert len(p_te) == len(y_test), f"alignment: {len(p_te)} vs {len(y_test)}"
print(f"Threshold {thr:.4f} on '{short}' scale | test rows = {len(y_test):,}")

# ---------- 4. Reload CSV, get test slice ----------
RAW_NAMES = ["Timestamp","From Bank","Account","To Bank","Account.1",
             "Amount Received","Receiving Currency","Amount Paid",
             "Payment Currency","Payment Format","Is Laundering"]

df = pd.read_csv(RAW_CSV, header=0, names=RAW_NAMES,
                 dtype={"From Bank":str,"Account":str,"To Bank":str,"Account.1":str})
df["source"] = df["From Bank"] + "_" + df["Account"]
df["target"] = df["To Bank"]   + "_" + df["Account.1"]
df = df.iloc[pd.to_datetime(df["Timestamp"], format="%Y/%m/%d %H:%M", utc=True)
             .argsort(kind="stable").to_numpy()].reset_index(drop=True)

va_end  = int(0.8 * len(df))
test_df = df.iloc[va_end:].reset_index(drop=True)
assert len(test_df) == len(p_te), f"row alignment: {len(test_df)} vs {len(p_te)}"
print(f"Test slice: {len(test_df):,} rows (starting at global row {va_end:,})")

# ---------- 5. Select alerts to explain ----------
flagged = np.where(p_te >= thr)[0]
if len(flagged) == 0:
    flagged = np.argsort(-p_te)[:N_ALERTS]
    print(f"⚠️  no rows crossed threshold; showing top {N_ALERTS} by score")
else:
    flagged = flagged[np.argsort(-p_te[flagged])][:N_ALERTS]

print(f"Explaining {len(flagged)} high-confidence alerts\n")

# ---------- 6. Evidence retriever ----------
from neo4j_graphrag.types import RetrieverResult, RetrieverResultItem

class AMLEvidenceRetriever:
    def __init__(self, drv, db="neo4j"):
        self.driver, self.database = drv, db
        self.cypher = """
        MATCH (s:Account {id: $source_id})-[t:TRANSACTION]->(r:Account {id: $target_id})
        WHERE t.timestamp = $timestamp
        WITH s, r, t
        ORDER BY abs(t.amount_paid - $amount_paid) ASC
        LIMIT 1
        CALL {
            WITH s, r
            MATCH (s)-[rel:TRANSACTION]->(other) WHERE other <> r
            RETURN collect(DISTINCT {
                account: other.id, amount: rel.amount_paid,
                format: rel.payment_format, ts: rel.timestamp
            })[0..8] AS s_sent
        }
        CALL {
            WITH s, r
            MATCH (other)-[rel:TRANSACTION]->(r) WHERE other <> s
            RETURN collect(DISTINCT {
                account: other.id, amount: rel.amount_paid,
                format: rel.payment_format, ts: rel.timestamp
            })[0..8] AS r_received
        }
        RETURN s.id AS source, r.id AS target,
               t.amount_paid AS amount_paid, t.amount_received AS amount_received,
               t.currency_paid AS currency_paid, t.currency_received AS currency_received,
               t.payment_format AS payment_format, t.timestamp AS time, t.label AS label,
               s_sent, r_received
        """

    def search(self, **kw):
        return self.get_search_results(**kw)

    def get_search_results(self, query_text=None, **kw):
        p = json.loads(query_text)
        with self.driver.session(database=self.database) as s:
            rec = s.run(self.cypher, **p).single()
        if not rec:
            return RetrieverResult(items=[], metadata={"error": "not found"})
        ctx = (
            f"FLAGGED TRANSACTION\n"
            f"  {rec['source']} → {rec['target']}\n"
            f"  Amount paid: {rec['amount_paid']} {rec['currency_paid']}\n"
            f"  Amount received: {rec['amount_received']} {rec['currency_received']}\n"
            f"  Format: {rec['payment_format']} | At: {rec['time']} | label={rec['label']}\n"
            f"\nSENDER's recent outgoing:\n{json.dumps(rec['s_sent'], indent=2, default=str)}\n"
            f"\nRECEIVER's recent incoming:\n{json.dumps(rec['r_received'], indent=2, default=str)}"
        )
        return RetrieverResult(items=[RetrieverResultItem(
            content=ctx, metadata={"source": rec["source"]})])

retriever = AMLEvidenceRetriever(driver)

# ---------- 7. Gemini client ----------
def _get_gemini_key():
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k: return k
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY", "")

GEMINI_KEY = _get_gemini_key()
if not GEMINI_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY not found.\n"
        "  Colab → 🔑 Secrets → add GEMINI_API_KEY → toggle 'Notebook access' ON → re-run")

try:
    from google import genai
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai"], check=True)
    from google import genai

client = genai.Client(api_key=GEMINI_KEY)

PROMPT = """You are a senior AML (Anti-Money Laundering) compliance analyst reviewing a transaction
flagged by our XGBoost + GNN detector. You must justify the alert using ONLY the graph evidence below.

EVIDENCE:
{context}

Write a concise 3-4 sentence explanation that:
1. States the suspicious flow (who sent what to whom, in what format).
2. Cites specific network context (the sender's other outflows, or the receiver's other inflows —
   look for pass-through patterns, rapid forwarding, structuring, or new accounts).
3. Ends with a verdict on its own line: [HIGH RISK: CONFIRMED] or [HIGH RISK: INCONCLUSIVE].

Rules:
- Do NOT invent accounts, amounts, or timestamps not present in the evidence.
- If the evidence is weak or ambiguous, say so and use INCONCLUSIVE.
- Do not mention "XGBoost", "GNN", or "evidence" — speak like a human analyst."""

def _call_gemini(context, retries=3):
    models = [GEMINI_MODEL] + [m for m in GEMINI_FALLBACK if m != GEMINI_MODEL]
    last_err = None
    for m in models:
        for attempt in range(retries):
            try:
                resp = client.models.generate_content(
                    model=m,
                    contents=PROMPT.format(context=context),
                    config={"temperature": 0.0, "max_output_tokens": 400},
                )
                return resp.text, m
            except Exception as e:
                last_err = e
                msg = str(e).lower()
                if "not found" in msg or "not supported" in msg:
                    break                               # try next model
                if "quota" in msg or "rate" in msg or "429" in msg:
                    wait = 2 ** attempt                 # 1s, 2s, 4s
                    print(f"    ⏳ rate-limited ({m}); retry in {wait}s …")
                    time.sleep(wait)
                    continue
                break                                   # non-retryable
    raise RuntimeError(f"all Gemini models failed: {type(last_err).__name__}: {last_err}")

print(f"Gemini ready — primary model: {GEMINI_MODEL}\n")

# ---------- 8. Generate explanations ----------
explanations = []
t0 = time.time()

for i, idx in enumerate(flagged):
    row = test_df.iloc[idx]
    query = json.dumps({
        "source_id":   str(row["source"]),
        "target_id":   str(row["target"]),
        "timestamp":   str(row["Timestamp"]),
        "amount_paid": float(row["Amount Paid"]),
    })

    print(f"--- Alert {i+1}/{len(flagged)} (score {p_te[idx]:.4f}) ---")
    res = retriever.search(query_text=query)

    if not res.items:
        print(f"  ❌ {row['source']} → {row['target']} not found in graph")
        explanations.append({
            "rank": i+1, "row": int(idx + va_end),
            "score": float(p_te[idx]),
            "source": str(row["source"]), "target": str(row["target"]),
            "error": "edge not found in Neo4j",
        })
        print()
        continue

    context = res.items[0].content
    try:
        answer, model_used = _call_gemini(context)
    except Exception as e:
        print(f"  ❌ Gemini failed: {e}")
        answer, model_used = f"[LLM unavailable: {e}]\n\nRAW EVIDENCE:\n{context}", "none"

    explanations.append({
        "rank": i+1,
        "row": int(idx + va_end),
        "score": float(p_te[idx]),
        "source": str(row["source"]),
        "target": str(row["target"]),
        "amount_paid": float(row["Amount Paid"]),
        "timestamp": str(row["Timestamp"]),
        "model": model_used,
        "explanation": answer,
    })
    print(answer)
    print()

elapsed = time.time() - t0

# ---------- 9. Save + summary ----------
out_path = f"{OUTPUT_DIR}/explanations.json"
with open(out_path, "w") as f:
    json.dump(explanations, f, indent=2, default=float)

try:
    open(f"{STATE_DIR}/graphrag_done.flag", "w").write("1")
except OSError:
    pass

ok = sum(1 for e in explanations
         if "explanation" in e and "[LLM unavailable" not in e.get("explanation", ""))

print("=" * 80)
print("GRAPH RAG — SUMMARY")
print("=" * 80)
print(f"Detection run:    {best_run}")
print(f"  Test AP:        {entry['ap_test']:.4f}")
print(f"  Threshold:      {thr:.5f}  (scale: {short})")
print(f"LLM model:        {GEMINI_MODEL}")
print(f"Alerts explained: {ok}/{len(explanations)}")
print(f"Elapsed:          {elapsed:.1f}s ({elapsed/max(len(flagged),1):.1f}s/alert)")
print(f"Saved:            {out_path}")
print(f"Drive flags:      neo4j_loaded ✓  pipeline_done ✓  graphrag_done ✓")

✅ Reusing existing Neo4j driver
Loaded results from 'structural': headline = blend (tabular + GNN, fitted on val)
Threshold -0.9300 on 'blend' scale | test rows = 1,015,669
Test slice: 1,015,669 rows (starting at global row 4,062,676)
Explaining 10 high-confidence alerts

Gemini ready — primary model: gemini-3.5-flash-lite

--- Alert 1/10 (score 4.8434) ---
Account `012_8010D4440` executed a suspicious ACH transfer of 916,645.83 US Dollars to receiver `0132554_80D31F5E0` on 2022/09/10 at 11:00. The sender's recent outgoing activity shows multiple fragmented ACH transfers (ranging from 134.36 to 2,321.52 US Dollars) back to the same account (`012_8010D4440`) and other entities in the days leading up to the flagged transaction. Meanwhile, the receiver exhibits frequent, recurring incoming transactions in small denominations (such as 852.19 and 1,430.53 US Dollars) via Cash and Cheque from a single source (`0017401_808E80F50`), contrasting sharply with the massive inbound ACH wire.

[HIGH

In [ ]:
# === In Colab — export the three artifacts for deployment ===
import json, os, shutil
from google.colab import files

# Find the trained model and feature list from the last run
# (these are the variables your pipeline leaves behind)
structural_dir = "/content/drive/MyDrive/aml/aml_outputs/structural"
out_dir = "/tmp/deploy_artifacts"
os.makedirs(out_dir, exist_ok=True)

# 1. Model file (already saved by run_experiment)
shutil.copy(f"{structural_dir}/xgb_tabular.json", f"{out_dir}/xgb_tabular.json")

# 2. Feature names — MUST match the order used at training time
#    `names` was built inside causal_features() and passed to run_experiment
#    If the variable isn't in scope, load it from the saved results.json
with open(f"{structural_dir}/results.json") as f:
    results = json.load(f)
feature_names = results["features"]          # ← saved by your run_experiment
json.dump(feature_names, open(f"{out_dir}/feature_names.json", "w"), indent=2)

# 3. Which features were log1p-transformed at training time
#    Re-derive from the same rules your pipeline used
logged = []
for nm in feature_names:
    if nm.startswith(("s_out_cnt", "s_in_cnt", "r_out_cnt", "r_in_cnt",
                      "s_out_sum", "s_in_sum", "r_out_sum", "r_in_sum",
                      "s_gap_", "r_gap_", "pair_",
                      "s_out_nuniq", "s_in_nuniq", "r_out_nuniq", "r_in_nuniq",
                      "s_age", "r_age", "log_amount",
                      "amt_vs_s_avg", "amt_vs_r_avg")):
        logged.append(nm)
json.dump(logged, open(f"{out_dir}/logged_features.json", "w"), indent=2)

# 4. Threshold (from your validated headline model)
thr = results[results["headline"]]["thr"]
json.dump({"threshold": float(thr), "headline": results["headline"]},
          open(f"{out_dir}/model_meta.json", "w"), indent=2)

print(f"Exported {len(feature_names)} feature names, {len(logged)} logged transforms")
print(f"Threshold: {thr}")
print("\nFiles to download:")
for f in os.listdir(out_dir):
    print(f"  {f} ({os.path.getsize(os.path.join(out_dir, f)):,} bytes)")

# 5. Download all four as individual files
for f in os.listdir(out_dir):
    files.download(os.path.join(out_dir, f))

Exported 80 feature names, 62 logged transforms
Threshold: -0.9300038486961864

Files to download:
  logged_features.json (1,104 bytes)
  xgb_tabular.json (9,679,565 bytes)
  feature_names.json (1,376 bytes)
  model_meta.json (92 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === In Colab — export a small graph sample for the Codespace demo ===
import json
import pandas as pd
from google.colab import files

# Load the previously-explained alerts (from Cell 5)
explanations = json.load(open(
    "/content/drive/MyDrive/aml/aml_outputs/explanations.json"))
alert_keys = set(
    (e["source"], e["target"], e["timestamp"])
    for e in explanations if "source" in e
)
print(f"Alert edges to anchor on: {len(alert_keys)}")

# Load the full raw CSV
RAW_NAMES = ["Timestamp","From Bank","Account","To Bank","Account.1",
             "Amount Received","Receiving Currency","Amount Paid",
             "Payment Currency","Payment Format","Is Laundering"]
df = pd.read_csv(
    "/content/drive/MyDrive/ibm aml/HI-Small_Trans.csv",
    header=0, names=RAW_NAMES,
    dtype={"From Bank":str,"Account":str,"To Bank":str,"Account.1":str},
)
df["source"] = df["From Bank"] + "_" + df["Account"]
df["target"] = df["To Bank"]   + "_" + df["Account.1"]

# Find the 10 alert edges in the full file
mask = df.apply(
    lambda r: (r["source"], r["target"], r["Timestamp"]) in alert_keys,
    axis=1,
)
alert_edges = df[mask].copy()
print(f"Found {len(alert_edges)} alert edges in raw CSV")

# 1-hop neighborhood around those alert accounts
alert_accounts = set(alert_edges["source"]) | set(alert_edges["target"])
neighborhood = df[
    df["source"].isin(alert_accounts) | df["target"].isin(alert_accounts)
].copy()
print(f"1-hop neighborhood: {len(neighborhood)} transactions")

# 500 random normal transactions for baseline density
random_normal = df[df["Is Laundering"] == 0].sample(n=500, random_state=42)
print(f"Random normal: {len(random_normal)} transactions")

# Combine + de-duplicate
sample = pd.concat([alert_edges, neighborhood, random_normal]).drop_duplicates()
print(f"Combined sample: {len(sample)} transactions")

# Export
out = sample[["source", "target", "Timestamp", "Amount Paid", "Amount Received",
              "Payment Currency", "Receiving Currency", "Payment Format", "Is Laundering"]].copy()
out.columns = ["source","target","timestamp","amount_paid","amount_received",
               "currency_paid","currency_received","payment_format","label"]
out.to_csv("/tmp/sample_graph.csv", index=False)
print(f"✅ Saved to /tmp/sample_graph.csv ({len(out)} rows)")
print(f"   Size: {len(out)*150/1024/1024:.1f} MB (approx)")

# Print one alert for use in the Codespace demo
demo = alert_edges.iloc[0]
print(f"\n📌 Demo transaction for Codespace test:")
print(json.dumps({
    "source": demo["source"],
    "target": demo["target"],
    "amount_paid": float(demo["Amount Paid"]),
    "timestamp": demo["Timestamp"],
}, indent=2))

files.download("/tmp/sample_graph.csv")

Alert edges to anchor on: 10
Found 12 alert edges in raw CSV
1-hop neighborhood: 823 transactions
Random normal: 500 transactions
Combined sample: 1323 transactions
✅ Saved to /tmp/sample_graph.csv (1323 rows)
   Size: 0.2 MB (approx)

📌 Demo transaction for Codespace test:
{
  "source": "012_8010D4440",
  "target": "0132554_80D31F5E0",
  "amount_paid": 916645.83,
  "timestamp": "2022/09/10 11:00"
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>